# A2 — Descarga reproducible de estaciones de carga desde OpenStreetMap

## 1. Propósito del entregable

Este entregable tiene como objetivo construir el inventario inicial de
estaciones de carga para vehículos eléctricos a partir de OpenStreetMap
(OSM), utilizando Overpass API como mecanismo de consulta.

La extracción se realiza mediante la etiqueta `amenity=charging_station`,
correspondiente a elementos de OpenStreetMap asociados con infraestructura
de carga para vehículos eléctricos.

La descarga se realiza de manera independiente para las dos ciudades
definidas como áreas de estudio de la investigación: Bogotá D.C. y
Medellín.

El proceso registra de manera explícita la consulta utilizada, el servidor Overpass que responde, la fecha y hora exacta de descarga, el número de elementos recuperados y el número de registros que finalmente se conservan después de aplicar la delimitación geográfica. El proceso permite además fijar un corte temporal sobre el estado de OpenStreetMap, de modo que la descarga sea reproducible en su resultado y no solo en su procedimiento.

El producto principal de este entregable es un archivo GeoJSON
independiente para cada ciudad.

Este inventario constituye la fuente de descubrimiento proveniente de
OpenStreetMap que será utilizada en las etapas posteriores del proyecto.

En este entregable no se realiza ningún contraste con OpenChargeMap,
emparejamiento entre fuentes, clasificación de estaciones ni cálculo de
distancias entre registros.

## 2. Alcance y delimitación

### 2.1. Fuente de datos

La fuente utilizada en este entregable es OpenStreetMap. La recuperación
de los datos se realiza mediante Overpass API.

OpenStreetMap se utiliza en esta etapa como fuente de descubrimiento de
estaciones de carga.

### 2.2. Criterio de selección

Se seleccionan exclusivamente los elementos que presentan la etiqueta:

`amenity=charging_station`

La consulta contempla los tres tipos de elementos que pueden representar
una estación dentro de OpenStreetMap:

- `node`;
- `way`;
- `relation`.

### 2.3. Áreas de estudio

La extracción se realiza de manera independiente para:

- Bogotá D.C.;
- Medellín.

Para cada ciudad se establece una bounding box en coordenadas geográficas
WGS84 (EPSG:4326). La bounding box se utiliza únicamente como área inicial
de consulta para Overpass.

Posteriormente, los resultados se recortan utilizando las geometrías
geográficas correspondientes a cada ciudad.

### 2.4. Delimitación geográfica

Para ambas ciudades se utiliza la capa nacional de municipios del IGAC, archivo `Municipios_Junio_2026.gpkg`, capa `Munpio`. El municipio de interés se selecciona por código DANE: 11001 para Bogotá y 05001 para Medellín.

Las geometrías se transforman de EPSG:9377 a WGS84 (EPSG:4326) para hacerlas compatibles con las coordenadas de las estaciones obtenidas desde OpenStreetMap.

### 2.5. Operaciones excluidas

Este entregable no realiza:

- contraste con OpenChargeMap;
- emparejamiento entre fuentes;
- comparación de nombres;
- comparación de operadores;
- cálculo de distancias entre estaciones;
- clasificación de estaciones;
- semáforos de confianza;
- reconciliación de atributos;
- análisis de vigencia.

Estas operaciones corresponden a entregables posteriores.

## 3. Librerías utilizadas

- `requests`: comunicación con los servidores Overpass API.
- `pandas`: organización y estructuración de los registros.
- `geopandas`: procesamiento de información geoespacial y exportación de los inventarios.
- `shapely`: construcción de las geometrías de bounding box para la comparación de delimitaciones.
- `folium`: mapas interactivos de comparación entre geometrías de delimitación.
- `hashlib`: huella SHA-256 del texto de la consulta.
- `json`: serialización de los campos anidados de OpenStreetMap.
- `time`: espera entre reintentos de descarga.
- `datetime` y `zoneinfo`: registro de la fecha y hora exacta de ejecución.
- `os`: interacción con el sistema operativo para la creación de directorios, manejo de rutas y gestión de archivos locales.

- `re`: uso de expresiones regulares para la limpieza, extracción y validación de patrones en cadenas de texto.

- `sys`: interacción con el intérprete de Python, útil para el control de errores, paso de argumentos o finalización de la ejecución.

- `unicodedata`: normalización de cadenas de texto, útil para remover acentos, tildes y caracteres especiales de los datos obtenidos.

- `difflib`: comparación de secuencias de texto (mediante SequenceMatcher) para encontrar similitudes o coincidencias parciales entre nombres o atributos.

- `numpy`: operaciones matemáticas avanzadas, manejo eficiente de arreglos numéricos y tratamiento de valores nulos o vacíos.

- `pyproj`: definición y transformación de Sistemas de Referencia de Coordenadas (CRS) para garantizar la precisión espacial de las geometrías.

- `matplotlib`: generación de gráficos estáticos y visualización de datos (usando pyplot, junto con Patch y Line2D para la personalización de leyendas y elementos visuales).

## 4. Configuración general

### 4.1. Identificación de la consulta

Se establece un `User-Agent` que identifica la actividad como parte de
una investigación académica.

### 4.2. Servidores Overpass

Se utilizan varios servidores Overpass como mecanismo de contingencia.

La consulta enviada es exactamente la misma independientemente del
servidor utilizado. Si un servidor no responde correctamente, se intenta
la ejecución con el siguiente servidor disponible.

El servidor que finalmente responde correctamente se registra en los
metadatos de la descarga.

### 4.3. Sistema de referencia

Las coordenadas de las estaciones y las delimitaciones geográficas se manejan en WGS84, correspondiente al EPSG:4326. Este sistema permite mantener una representación geográfica homogénea durante la consulta y el recorte espacial, y es el que se emplea en la exportación de los archivos GeoJSON.

Adicionalmente se declara un sistema de referencia proyectado en metros, MAGNA-SIRGAS / Origen Nacional (EPSG:9377), que no interviene en la descarga y se utiliza únicamente en las verificaciones independientes que requieren medir distancias en unidades métricas.

# Librerias

In [1]:
import hashlib
import json
import os
import re
import sys
import time
import unicodedata
from datetime import datetime
from difflib import SequenceMatcher
from zoneinfo import ZoneInfo

# Librerías de terceros
import folium
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyproj
import requests
import shapely
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from shapely.geometry import box

# Constantes de configuración

Este bloque no ejecuta ninguna operación: define los parámetros que gobiernan la descarga. Se declaran de manera centralizada y explícita para que cualquier persona que reejecute el cuaderno pueda identificar, en un solo lugar, bajo qué condiciones se realizó la extracción.

## 5.1. Identificación de la consulta

USER_AGENT declara la identidad del cliente que consulta Overpass API. La política de uso de la infraestructura de Overpass solicita que las consultas automatizadas se identifiquen; la cadena declara el nombre del proyecto, su versión y su naturaleza académica.

## 5.2. Servidores Overpass

OVERPASS_SERVERS contiene la lista ordenada de servidores espejo que pueden responder la consulta. El orden no es arbitrario: overpass-api.de ocupa la primera posición porque es el único de la lista que garantiza el servicio de datos históricos, requerido cuando se fija un corte temporal (sección 5.6). Los tres restantes actúan como contingencia ante indisponibilidad del primero.

## 5.3. Zona horaria y sistemas de referencia

TIMEZONE fija la zona horaria America/Bogota para el registro de la fecha y hora de descarga. Esto evita que la marca temporal quede en la zona horaria de la máquina donde se ejecute el cuaderno, lo que la haría ambigua.

Se declaran dos sistemas de referencia con propósitos distintos:

CRS_WGS84 (EPSG:4326): sistema geográfico en grados. Es el sistema en que OpenStreetMap entrega las coordenadas y en el que se realiza el recorte espacial y la exportación de los archivos GeoJSON.
CRS_METRICO (EPSG:9377, MAGNA-SIRGAS / Origen Nacional): sistema proyectado en metros, oficial del IGAC para Colombia. No se usa para la descarga; se emplea únicamente en las verificaciones independientes de las pruebas de aceptación, donde se requiere medir distancias en unidades métricas y no en grados.

## 5.4. Criterio de selección

ETIQUETA_OSM registra la etiqueta de OpenStreetMap que define el criterio de inclusión: amenity=charging_station.

5.5. Control de tiempos y reintentos

Se declaran dos tiempos de espera que operan en niveles diferentes y no deben confundirse:

TIMEOUT_OVERPASS_HTTP (180 segundos): tiempo máximo que el cliente espera una respuesta de la conexión HTTP antes de abandonarla.
TIMEOUT_OVERPASS_QUERY (120 segundos): tiempo máximo que el servidor Overpass dedica a resolver la consulta. Se interpola dentro del texto de la consulta misma.

El primero es mayor que el segundo de manera deliberada: la conexión debe sobrevivir lo suficiente para recibir el mensaje de error que el servidor emite cuando agota su propio límite.

MAX_REINTENTOS y ESPERA_REINTENTO gobiernan el comportamiento ante fallos transitorios: número de reintentos adicionales sobre un mismo servidor y segundos de espera entre ellos.

## 5.6. Corte temporal de OpenStreetMap

FECHA_CORTE_OSM es el parámetro que determina si la descarga es reproducible en sentido estricto.

OpenStreetMap es una base de datos viva: el mismo bounding box consultado en dos momentos distintos puede devolver conjuntos distintos de estaciones. Overpass API permite solicitar el estado del mapa en un instante pasado mediante una attic query.

El corte quedó fijado en el 29 de agosto de 2026, y la razón no proviene de OpenStreetMap. OpenChargeMap no presta servicio de consulta histórica: su API devuelve siempre el estado actual del registro, de modo que la única manera de fijar un estado reproducible es congelar el archivo descargado. Fijada esa fecha por necesidad en una fuente, corresponde aplicarla también a la otra: si OpenStreetMap se consultara al estado vigente y OpenChargeMap a un archivo del 29 de agosto, el emparejamiento del Bloque B compararía dos momentos distintos del mundo y cualquier discrepancia entre fuentes quedaría contaminada por el desfase temporal.

El instante se declara como el final del 29 de agosto en hora de Bogotá. Overpass exige la fecha en UTC, y como Bogotá es UTC−5, el mismo instante se escribe 2026-08-30T04:59:59Z. Ambas formas se declaran en el código para que la conversión no tenga que rehacerse cada vez que se lea.

No todos los espejos prestan el servicio de datos históricos, y uno que no lo preste ignora la directiva y devuelve el estado actual sin emitir error. El cuaderno atiende esa posibilidad por dos vías: con el corte fijado la descarga se restringe a los espejos que sí lo garantizan, y el corte declarado por el servidor se contrasta contra el solicitado, tratándose la discrepancia como fallo del servidor y no como resultado aceptable.

## 5.7. Atribución de licencia

ATRIBUCION_ODBL contiene el texto de atribución exigido por la licencia Open Database License bajo la cual se distribuyen los datos de OpenStreetMap. La cláusula de atribución de la ODbL obliga a acompañar los datos derivados con el reconocimiento de la fuente; la constante se declara aquí para incorporarse a los metadatos de cada archivo exportado.

In [2]:
USER_AGENT = (
    "tesis-estaciones-carga-electrica/1.0 "
    "(investigacion-academica)"
)

# overpass-api.de va primero porque es el único que garantiza
# datos históricos, necesarios cuando se fija FECHA_CORTE_OSM.
OVERPASS_SERVERS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
    "https://overpass.openstreetmap.fr/api/interpreter",
]

OVERPASS_SERVERS_HISTORICOS = [
    "https://overpass-api.de/api/interpreter",
]

TIMEZONE = ZoneInfo("America/Bogota")

CRS_WGS84 = "EPSG:4326"

# MAGNA-SIRGAS / Origen Nacional. Métrico, oficial IGAC.
# Se usa para las verificaciones independientes en metros
# de las pruebas de aceptación.
CRS_METRICO = "EPSG:9377"

ETIQUETA_OSM = "amenity=charging_station"

# Dos timeouts distintos: uno gobierna la conexión HTTP,
# el otro se interpola dentro del texto de la consulta.
TIMEOUT_OVERPASS_HTTP = 180

TIMEOUT_OVERPASS_QUERY = 120

MAX_REINTENTOS = 2

ESPERA_REINTENTO = 5

# ----------------------------------------------------------------------------
# Corte temporal del inventario
# ----------------------------------------------------------------------------
# El corte es el 29/08/2026 y es común a las dos fuentes del Bloque A.
# La razón no proviene de OpenStreetMap sino de OpenChargeMap: OCM no
# admite consultas históricas, de modo que su archivo debe congelarse
# en disco a una fecha. Para que ambas fuentes describan el mismo
# estado del mundo, OSM se corta en ese mismo instante mediante attic
# query. Cortar cada fuente en una fecha distinta haría que el
# emparejamiento del Bloque B comparara dos momentos diferentes.
#
# El instante se declara como el final del 29/08/2026 en hora de
# Bogotá. Overpass exige UTC y Bogotá es UTC-5, de modo que el mismo
# instante se escribe 2026-08-30T04:59:59Z. Las dos formas se
# declaran juntas para no tener que rehacer la conversión mentalmente.

FECHA_CORTE_LOCAL = "2026-08-29"

FECHA_CORTE_OSM = "2026-08-30T04:59:59Z"

# Prefijo del archivo crudo congelado. La fecha que entra en el
# nombre es la del CORTE, no la del día en que se ejecuta el
# cuaderno: el archivo contiene el estado del mapa al 29/08/2026.
PREFIJO_CRUDO = "A2_osm_crudo"

ATRIBUCION_ODBL = (
    "Datos de OpenStreetMap, © colaboradores de OpenStreetMap, "
    "disponibles bajo la Open Database License (ODbL). "
    "https://www.openstreetmap.org/copyright"
)

# 6. Áreas de estudio y delimitación geográfica

Este bloque declara tres objetos y documenta una sustitución de insumo ya
ejecutada.

`LIMITE_IGAC` contiene la delimitación adoptada. `LIMITES_SUSTITUIDOS`
conserva declarados los dos límites que se emplearon hasta la versión
anterior, fuera del circuito de análisis. `AREAS_ESTUDIO` reúne los
parámetros de extracción de cada ciudad y apunta, en ambas, al límite
adoptado.

La sustitución no se ejecutó en silencio. Se declaró la fuente candidata,
se comparó geométricamente contra las vigentes, y solo con esa comparación
documentada se decidió el reemplazo. Las secciones que siguen reconstruyen
ese proceso en el orden en que ocurrió: qué había, qué defecto se detectó,
qué se evaluó, qué midió la comparación y qué se decidió.

## 6.1. El punto de partida: dos límites de procedencia distinta

La versión anterior del cuaderno recortaba cada ciudad con un archivo
propio, y las dos procedencias no eran equivalentes.

**Bogotá** se recortaba con `muni.gpkg`, capa `Muni`, descargada del portal
de IDECA, la infraestructura de datos espaciales del Distrito Capital. Es
una fuente administrativa distrital, ajena a OpenStreetMap.

**Medellín** se recortaba con `medellin.gpkg`, capa `limite_medellin`,
generada con `osmnx.geocode_to_gdf('Medellín, Antioquia, Colombia')`, es
decir, resolviendo el nombre del municipio contra Nominatim, que consulta
la propia base de OpenStreetMap.

La estructura del código era idéntica en ambas ciudades. La naturaleza del
insumo no.

## 6.2. Los defectos detectados

**Medellín: dependencia circular.** El límite provenía de OpenStreetMap, la
misma fuente de la que se extraen las estaciones. El filtro espacial no
constituía por tanto una verificación independiente: se comprobaba que
datos de OpenStreetMap cayeran dentro de un polígono de OpenStreetMap, lo
cual no aporta corroboración alguna. Una estación mal ubicada en la base
colaborativa no tenía forma de ser detectada por un límite derivado de esa
misma base.

A esto se sumaba que el resultado dependía de la versión de la relación
administrativa vigente al momento de la consulta, versión que no quedó
registrada. La descarga no era reproducible ni siquiera contra sí misma.

**Bogotá: trazabilidad incompleta.** El límite de IDECA sí era
independiente de OpenStreetMap, de modo que el filtro espacial sí era una
verificación externa. El defecto era de otro orden: la ficha exacta del
recurso no quedó registrada al momento de la descarga. No podía afirmarse
con precisión qué versión del límite distrital se había empleado.

**Ambas ciudades: asimetría metodológica.** El defecto de fondo no era ni
uno ni otro por separado, sino que las dos áreas de estudio se estaban
tratando con estándares distintos. Cualquier diferencia observada entre
Bogotá y Medellín podía provenir de los datos o del método, y no había
forma de distinguirlo.

## 6.3. La fuente evaluada: capa nacional de municipios del IGAC

Frente a esos defectos se evaluó la capa nacional de municipios publicada
por el Instituto Geográfico Agustín Codazzi, obtenida del portal Colombia
en Mapas. Es un único archivo GeoPackage con 1.105 registros —la totalidad
de los municipios del país— en el sistema de referencia MAGNA-SIRGAS /
Origen Nacional (EPSG:9377), versión de junio de 2026, descargado
manualmente el 09/09/2026.

**Independencia respecto de OpenStreetMap.** El campo
`independiente_de_osm` vale `True`. Es un producto cartográfico oficial
elaborado por la autoridad geográfica nacional, sin relación con la base
colaborativa de la que se extraen las estaciones. Esta propiedad es la que
corrige el defecto de Medellín, y la razón principal por la que la fuente
se evaluó.

**Carácter oficial.** La capa del IGAC es la delimitación producida por la
autoridad cartográfica nacional, con vigencia declarada y criterios de
elaboración públicos. Frente a ella, el límite de IDECA es una
delimitación distrital y el de Medellín una reconstrucción colaborativa sin
carácter oficial. Si las tres geometrías discrepan sobre dónde termina un
municipio, la del IGAC es la que tiene respaldo institucional para
dirimirlo. Eso la convierte en la referencia contra la cual medir las otras
dos, y no en una alternativa equivalente.

**Selección por código y no por nombre.** El campo `columna_codigo` y el
subdiccionario `codigos_dane` establecen que el municipio se identifica por
su código DANE: 11001 para Bogotá y 05001 para Medellín. La decisión no es
cosmética. En esta capa el municipio 11001 figura con la denominación
"Bogotá, D.C.", con coma y abreviatura. Una búsqueda por nombre tendría que
anticipar esa forma exacta, además de la tilde y la variabilidad de
mayúsculas, y fallaría de manera silenciosa —devolviendo cero registros—
ante cualquier discrepancia. El código DANE es un identificador estable y
normalizado, y por eso constituye la llave de selección.

## 6.4. La restricción declarada por la propia fuente

La capa incluye un campo `Restriccion` en el que la entidad productora
advierte que el producto no es apropiado para la ubicación exacta de puntos
ni para el cálculo de áreas y distancias. Esa advertencia se transcribe
literalmente en el campo `restriccion_declarada_por_la_fuente`.

La transcripción no es un formalismo. El uso que este trabajo hace de la
capa es precisamente la ubicación de puntos: se determina si cada estación
de carga cae dentro o fuera del polígono municipal. La fuente advierte, en
otras palabras, que su geometría no tiene la exactitud necesaria para
sostener con certeza ese juicio en casos límite.

La advertencia no invalida el uso. La capa del IGAC es la delimitación
municipal oficial de Colombia y no existe una alternativa abierta de mayor
exactitud declarada. Lo que la advertencia impone es una limitación sobre
lo que puede afirmarse: una estación situada a decenas de metros del borde
municipal queda clasificada por una geometría cuya propia entidad
productora no garantiza a esa escala.

De ahí se derivan dos consecuencias operativas, ambas implementadas:

1. El recorte espacial emplea el predicado `intersects` y no `within`, de
   modo que los puntos situados exactamente sobre la frontera se conservan.
   Ante un dato cuya exactitud en el borde no está garantizada por su
   productor, el criterio inclusivo es el prudente.
2. Si alguna estación resultara conservada por esa razón, corresponde
   identificarla individualmente y reportarla como caso indeterminado, en
   lugar de asignarla sin más a una u otra categoría.

Registrar la restricción dentro del código, y no únicamente en el informe,
hace que la limitación viaje junto al dato y no dependa de que alguien
recuerde mencionarla.

## 6.5. La comparación que sustentó la decisión

Antes de sustituir se compararon las tres geometrías de cada ciudad —bbox
de consulta, límite vigente y capa del IGAC— sobre dos frentes. El primero,
geométrico, se documenta en la sección siguiente y sus resultados se
resumen aquí. El segundo, sobre el inventario descargado, solo puede
ejecutarse después de la descarga y se retoma en 6.7.

Para cada par se calcularon el área de cada geometría, el área de su
intersección, el índice de Jaccard como medida global de solape, el área
que pertenece a una y no a la otra, y las relaciones de contención en
ambos sentidos. Todas las medidas de superficie se tomaron en el sistema
métrico EPSG:9377 y no en grados, porque un área calculada sobre
coordenadas geográficas no tiene unidades interpretables.

**Cuánto difiere el límite vigente del oficial.**

| | Bogotá | Medellín |
|---|---|---|
| Área IGAC | 1.618,2 km² | 373,4 km² |
| Área del límite sustituido | 1.633,1 km² | 374,6 km² |
| Índice de Jaccard | 0,9763 | 0,9835 |
| Solo en el límite sustituido | 27,0 km² | 3,7 km² |
| Solo en el IGAC | 12,0 km² | 2,5 km² |

La lectura es directa: las geometrías coinciden en más del 97% de su
superficie en Bogotá y en más del 98% en Medellín. **La sustitución cambia
la procedencia y la independencia del insumo, no el área de estudio.** Ese
es el resultado que autoriza el reemplazo: si las geometrías hubieran
diferido de manera sustantiva, el cambio habría alterado el objeto de
estudio y no solo su respaldo documental, y habría exigido rehacer los
bloques anteriores.

**Cuál de las dos geometrías es el recorte vinculante.** La comparación
respondió además una pregunta que la versión anterior del cuaderno daba por
sentada sin verificarla.

En **Bogotá**, ninguna de las dos geometrías contiene a la otra. 1.010,2
km² del Distrito quedan fuera del bbox —corresponden a la localidad de
Sumapaz, exclusión deliberada y coherente con la cobertura del Modelo
Digital de Terreno de IDECA, que es zona urbana— y 338,2 km² del área
consultada quedan fuera del municipio, repartidos entre once municipios
vecinos: Soacha, Funza, Cota, Tenjo, Mosquera, Chipaque, Ubaque, Madrid, La
Calera, Chía y Sibaté. El recorte efectivo es por tanto la **intersección**
de ambas geometrías, y el filtro por límite es recorte vinculante y no mera
validación de consistencia, contra lo que afirmaba la versión anterior de
este criterio.

En **Medellín**, el límite municipal está contenido por completo en el bbox
—0,0 km² del municipio quedan fuera del área consultada—, de modo que el
recorte vinculante es el límite. El bbox alcanza 268,6 km² de municipios
del Área Metropolitana —Bello, Copacabana, Envigado, Itagüí, Sabaneta,
entre otros— que el filtro descarta.

## 6.6. La decisión

Con esa evidencia se adoptó la capa del IGAC como delimitación para ambas
ciudades. Tres razones, en orden de peso:

**Independencia.** Corrige el defecto de fondo. El filtro espacial de
Medellín pasa de ser un recorte de consistencia interna a una verificación
contra una fuente que no participó en el registro de las estaciones. Es la
condición que hace que el recorte signifique algo.

**Trazabilidad.** Corrige el defecto de Bogotá. La procedencia del insumo
queda declarada en el código —entidad, URL, fecha de descarga, sistema de
referencia de origen, método de obtención y restricción declarada— y se
propaga a los metadatos de cada archivo exportado.

**Simetría.** Ambas ciudades pasan a recortarse con la misma capa, en
igualdad de condiciones. Cualquier diferencia observada entre Bogotá y
Medellín en los bloques siguientes proviene de los datos y no del método.

**El costo.** La sustitución mueve 27,0 km² fuera del área de estudio en
Bogotá y 3,7 km² en Medellín, e incorpora 12,0 km² y 2,5 km²
respectivamente. Sobre superficies de 1.618 y 373 km², la magnitud está por
debajo del umbral en que el cambio tendría consecuencias metodológicas.

**Lo que se conserva.** Los dos límites anteriores quedan declarados en
`LIMITES_SUSTITUIDOS`, con su procedencia y el motivo de la sustitución,
fuera del circuito de análisis. Solo los leen las celdas de comparación y
de mapas. Esa conservación cumple dos funciones: deja registro de la
versión anterior y permite reproducir la comparación que justificó el
cambio sin recuperar archivos de otra parte.

## 6.7. El efecto sobre el inventario

El frente geométrico responde cuánto cambia el área de estudio. El frente
práctico responde cuánto cambia el inventario, y solo puede medirse tras la
descarga. Consiste en clasificar cada estación según caiga dentro del bbox,
dentro del límite municipal, o dentro del primero pero fuera del segundo
—caso que corresponde a una estación perteneciente a otro municipio y que
debe descartarse.

Los conteos se reportan en la auditoría de la sección 21. Con el límite
adoptado, el recorte descartó 2 de 44 estaciones en Bogotá y 3 de 14 en
Medellín. La cifra de Medellín es proporcionalmente mayor y tiene
explicación geométrica: el bbox alcanza los municipios del Área
Metropolitana, densamente conurbados con la ciudad.

La verificación de que el recorte se aplicó correctamente no descansa en el
propio cruce espacial que lo produjo. La prueba de aceptación 8 reproyecta
estaciones y límite a EPSG:9377 y mide la distancia de cada punto al
polígono, exigiendo que sea cero dentro de una tolerancia de un milímetro.
Es un método distinto —medida métrica en lugar de predicado topológico en
grados— aplicado al mismo problema.

## 6.8. Cómo queda declarado en el código

**`LIMITE_IGAC`** reúne la ruta del archivo, la capa, la columna de código,
los códigos DANE de ambas ciudades, la marca de independencia respecto de
OpenStreetMap y el subdiccionario de procedencia, que incluye la
restricción declarada por la fuente. Es el objeto que el ciclo de ejecución
lee para cargar el límite de cada ciudad.

**`LIMITES_SUSTITUIDOS`** conserva, por ciudad, el archivo, la capa, la
marca de independencia, el motivo de la sustitución y la procedencia del
insumo anterior. Ninguna función del pipeline lo lee.

**`AREAS_ESTUDIO`** conserva, por ciudad, la denominación, el código DANE,
el bbox de consulta, el `criterio_espacial` que justifica la delimitación
adoptada, y `limite`, que apunta a `LIMITE_IGAC`.

El código DANE de cada ciudad queda declarado en dos lugares: en
`AREAS_ESTUDIO`, donde documenta la configuración de cada área de estudio,
y en `codigos_dane` dentro de `LIMITE_IGAC`, que es el que el ciclo de
ejecución y las celdas de comparación consultan efectivamente. Ambas
declaraciones deben mantenerse consistentes entre sí: si se incorporara una
tercera ciudad o cambiara un código, hay que actualizar los dos lugares.

Ambas ciudades comparten el mismo objeto `LIMITE_IGAC`, no una copia. La
capa nacional única reemplaza dos archivos de procedencia distinta —uno
distrital, otro derivado de OpenStreetMap— por una fuente oficial e
independiente de la fuente de estaciones en los dos casos.

In [3]:
# ----------------------------------------------------------------------------
# Delimitación adoptada: capa oficial de municipios del IGAC
# ----------------------------------------------------------------------------
# Sustituye los límites empleados hasta la versión anterior
# (IDECA para Bogotá, OpenStreetMap/Nominatim para Medellín).
# La sustitución se sustenta en la comparación geométrica de la
# sección siguiente: IoU de 0,976 en Bogotá y 0,984 en Medellín
# frente a los límites anteriores, de modo que el cambio afecta la
# procedencia y la independencia del insumo, no el área de estudio.
# Decision explicada despues del mapa de Medellín

LIMITE_IGAC = {

    "archivo": "../Municipios_Junio_2026.gpkg",

    "capa": "Munpio",

    "columna_codigo": "MpCodigo",

    # Se filtra por código DANE y NO por nombre: en esta capa
    # Bogotá figura como "Bogotá, D.C.", con coma y abreviatura.
    "codigos_dane": {
        "Bogota": "11001",
        "Medellin": "05001",
    },

    # Fuente administrativa nacional, ajena a OpenStreetMap para
    # ambas ciudades. El filtro espacial es verificación externa.
    "independiente_de_osm": True,

    "procedencia": {

        "entidad": "IGAC — Instituto Geográfico Agustín Codazzi",

        "url": "https://www.colombiaenmapas.gov.co/",

        "fecha_descarga": "09/09/2026",

        "crs_origen": "EPSG:9377",

        "obtencion": (
            "Descarga manual desde el portal Colombia en Mapas. "
            "Capa nacional de municipios, versión junio de 2026, "
            "1.105 registros."
        ),

        "restriccion_declarada_por_la_fuente": (
            "No es apropiada su aplicación para la ubicación exacta "
            "de puntos ni para el cálculo de áreas y distancias"
        ),
    },
}


# ----------------------------------------------------------------------------
# Límites sustituidos
# ----------------------------------------------------------------------------
# Se conservan declarados, fuera del circuito de análisis, como
# registro de la versión anterior y para poder reproducir la
# comparación que justificó la sustitución.

LIMITES_SUSTITUIDOS = {

    "Bogota": {

        "archivo": "../muni.gpkg",

        "capa": "Muni",

        "independiente_de_osm": True,

        "motivo_sustitucion": (
            "Trazabilidad incompleta: la ficha exacta del recurso no "
            "quedó registrada al momento de la descarga. El límite "
            "era independiente de OpenStreetMap, de modo que la "
            "sustitución no corrige un defecto de independencia sino "
            "de documentación del insumo."
        ),

        "procedencia": {

            "entidad": (
                "IDECA — Infraestructura de Datos Espaciales "
                "para el Distrito Capital"
            ),

            "url": (
                "https://www.ideca.gov.co/recursos/mapas/"
                "municipio-bogota-dc"
            ),

            "fecha_descarga": "01/09/2026",

            "obtencion": (
                "Descarga manual desde el portal IDECA. La ficha "
                "exacta del recurso no quedó registrada."
            ),
        },
    },

    "Medellin": {

        "archivo": "../medellin.gpkg",

        "capa": "limite_medellin",

        "independiente_de_osm": False,

        "motivo_sustitucion": (
            "Dependencia circular: el límite provenía de "
            "OpenStreetMap, la misma fuente de la que se extraen "
            "las estaciones, de modo que el filtro espacial no "
            "constituía verificación independiente. Es el defecto "
            "que la sustitución corrige."
        ),

        "procedencia": {

            "entidad": (
                "OpenStreetMap — relación administrativa de "
                "Medellín, resuelta mediante Nominatim"
            ),

            "url": "https://nominatim.openstreetmap.org",

            "fecha_descarga": "01/09/2026",

            "obtencion": (
                "Generado con osmnx.geocode_to_gdf('Medellín, "
                "Antioquia, Colombia') y exportado a GPKG. El "
                "resultado depende de la versión de la relación "
                "vigente al momento de la consulta, no registrada."
            ),
        },
    },
}


# ----------------------------------------------------------------------------
# Áreas de estudio
# ----------------------------------------------------------------------------

AREAS_ESTUDIO = {

    "Bogota": {

        "nombre": "Bogotá D.C.",

        "codigo_dane": "11001",

        # Sobre de consulta para Overpass, NO recorte final.
        # Excluye Sumapaz por el sur, pero se desborda por el
        # occidente sobre municipios vecinos; ver 'criterio_espacial'.
        "bbox": {
            "oeste": -74.2235,
            "sur": 4.4711,
            "este": -74.0102,
            "norte": 4.8331,
        },

        "criterio_espacial": (
            "El recorte efectivo es la INTERSECCIÓN del bbox con el "
            "límite municipal, porque ninguno de los dos contiene al "
            "otro. El bbox aporta la exclusión de Sumapaz: 1.010,2 "
            "km² del Distrito quedan fuera del área consultada, "
            "coherente con la cobertura del Modelo Digital de "
            "Terreno de IDECA, que es zona urbana. El límite del "
            "IGAC aporta la exclusión de los municipios vecinos que "
            "el bbox alcanza —Soacha, Funza, Cota, Tenjo, Mosquera, "
            "Chipaque, Ubaque, Madrid, La Calera, Chía y Sibaté— y "
            "que suman 338,2 km² del área consultada. El filtro por "
            "límite es por tanto recorte vinculante y no mera "
            "validación de consistencia, contra lo que afirmaba la "
            "versión anterior de este criterio."
        ),

        "limite": LIMITE_IGAC,
    },

    "Medellin": {

        "nombre": "Medellín",

        "codigo_dane": "05001",

        # El límite municipal está íntegramente contenido en este
        # bbox, de modo que el recorte vinculante es el límite.
        "bbox": {
            "oeste": -75.7193,
            "sur": 6.1629,
            "este": -75.4718,
            "norte": 6.3748,
        },

        "criterio_espacial": (
            "El límite municipal del IGAC está contenido por "
            "completo en el bbox —0,0 km² del municipio quedan fuera "
            "del área consultada—, de modo que el recorte vinculante "
            "es el límite. El bbox alcanza municipios vecinos del "
            "Área Metropolitana —Bello, Copacabana, Envigado, "
            "Itagüí, Sabaneta, entre otros— que suman 268,6 km² y "
            "que el filtro por límite descarta. "
            "SUSTITUCIÓN: el límite empleado antes provenía de "
            "OpenStreetMap vía Nominatim, la misma fuente de la que "
            "se extraen las estaciones, por lo que el filtro "
            "espacial no constituía verificación independiente. Con "
            "la capa del IGAC el filtro pasa a ser verificación "
            "externa, en igualdad de condiciones con Bogotá."
        ),

        "limite": LIMITE_IGAC,
    },
}

# Validación de los bounding box

Antes de emplear los bbox declarados en AREAS_ESTUDIO (tanto para consultar Overpass como para construir geometrías de comparación) se verifica que sean estructural y geográficamente válidos. La función validar_bbox cumple esa función y se ejecuta sobre todas las ciudades inmediatamente después de la declaración.

La validación se ubica en este punto de manera deliberada. Un bbox mal formado no produce un error visible: si dos coordenadas están invertidas, la construcción del rectángulo tiene éxito y devuelve una geometría perfectamente válida, solo que en el lugar equivocado. El fallo se manifestaría mucho más adelante, como una descarga vacía o un recorte incorrecto, y el diagnóstico exigiría remontar toda la cadena. Validar en el punto de declaración convierte un error silencioso y de manifestación tardía en un error inmediato y localizado.

## 7.1. Comportamiento ante el error

La función interrumpe la ejecución mediante ValueError en lugar de devolver un valor de falla o emitir una advertencia. La decisión es coherente con la naturaleza del problema: un bbox inválido no admite continuación razonable, y todo lo que se construya a partir de él estará mal. Un aviso que el cuaderno pudiera ignorar dejaría abierta la posibilidad de que el error se propague sin ser advertido.

Cada mensaje de error incorpora el nombre de la ciudad afectada, de modo que al ejecutarse el ciclo sobre varias áreas de estudio quede claro cuál falló.

Cuando todas las comprobaciones pasan, la función devuelve True.

## 7.2. Comprobaciones de estructura

La primera verificación es de completitud. Se compara el conjunto de claves presentes contra el conjunto requerido (oeste, sur, este, norte) mediante diferencia de conjuntos, y se reportan todas las claves ausentes en un solo mensaje.

La formulación por conjuntos tiene dos ventajas sobre una comprobación clave por clave. Informa de todas las ausencias simultáneamente, en lugar de detenerse en la primera. Y es independiente del orden en que las claves aparezcan en el diccionario.

## 7.3. Comprobaciones de rango

Se verifica que cada coordenada caiga dentro del dominio del sistema de referencia WGS84: las longitudes oeste y este entre −180 y 180 grados, y las latitudes sur y norte entre −90 y 90.

La validación de rango es, en consecuencia, una red parcial. Detecta el error cuando produce valores fuera de dominio, pero no puede detectarlo cuando ambos valores permanecen dentro de sus rangos respectivos. Las comprobaciones de la sección siguiente cubren parte de ese vacío.

## 7.4. Comprobaciones de consistencia interna

Las dos últimas verificaciones exigen que oeste sea estrictamente menor que este y que sur sea estrictamente menor que norte. Un rectángulo geográfico bien formado en el hemisferio occidental y septentrional debe satisfacer ambas relaciones.

Estas comprobaciones capturan la inversión de un par de coordenadas, que es el error que la validación de rango deja pasar. Detectan además el caso degenerado de igualdad, que produciría un rectángulo de ancho o alto nulo y, en consecuencia, un área de consulta vacía.

La comparación es estricta y no admite igualdad, precisamente para excluir ese caso degenerado.

Limitación que conviene declarar. La condición oeste < este no es universalmente correcta: un bbox que cruce el antimeridiano tiene oeste mayor que este, y esta función lo rechazaría. La restricción es admisible en este trabajo porque ambas ciudades de estudio se sitúan holgadamente lejos de esa longitud, pero la función no es reutilizable sin ajuste fuera de ese supuesto.

## 7.5. Lo que esta validación no comprueba

La función verifica que el bbox sea un rectángulo geográfico bien formado. No verifica que sea el rectángulo correcto.

En particular, no comprueba que el bbox cubra efectivamente la ciudad declarada, ni que guarde alguna relación determinada con el límite municipal. Un rectángulo situado en otro continente pasaría todas estas comprobaciones sin objeción.

Esa segunda clase de verificación (la correspondencia entre el bbox y la geometría de la ciudad) corresponde a la comparación geométrica documentada en la sección anterior, que mide el solape entre ambas y determina cuál de las dos constituye el recorte vinculante. Las dos validaciones son complementarias y operan en niveles distintos: esta comprueba la forma del dato, aquella comprueba su pertinencia.

In [4]:
def validar_bbox(bbox, nombre_ciudad):
    """
    Valida la estructura y los valores geográficos
    del bounding box de una ciudad.
    """

    campos_requeridos = {
        "oeste",
        "sur",
        "este",
        "norte",
    }

    campos_faltantes = (
        campos_requeridos
        - set(bbox.keys())
    )

    if campos_faltantes:
        raise ValueError(
            f"{nombre_ciudad}: faltan campos del BBOX: "
            f"{campos_faltantes}"
        )

    oeste = bbox["oeste"]
    sur = bbox["sur"]
    este = bbox["este"]
    norte = bbox["norte"]

    if not (-180 <= oeste <= 180):
        raise ValueError(
            f"{nombre_ciudad}: longitud oeste inválida."
        )

    if not (-180 <= este <= 180):
        raise ValueError(
            f"{nombre_ciudad}: longitud este inválida."
        )

    if not (-90 <= sur <= 90):
        raise ValueError(
            f"{nombre_ciudad}: latitud sur inválida."
        )

    if not (-90 <= norte <= 90):
        raise ValueError(
            f"{nombre_ciudad}: latitud norte inválida."
        )

    if oeste >= este:
        raise ValueError(
            f"{nombre_ciudad}: oeste debe ser menor que este."
        )

    if sur >= norte:
        raise ValueError(
            f"{nombre_ciudad}: sur debe ser menor que norte."
        )

    return True

# --- Ejecución sobre las dos ciudades ---
for clave, ciudad in AREAS_ESTUDIO.items():
    validar_bbox(ciudad["bbox"], ciudad["nombre"])

print("BBOX validados para todas las ciudades.")

BBOX validados para todas las ciudades.


# Mapa interactivo con las tres geometrías

In [5]:
CIUDAD = "Bogota"
cfg = AREAS_ESTUDIO[CIUDAD]

# --- Capa adoptada: IGAC ---
igac = gpd.read_file(
    LIMITE_IGAC["archivo"],
    layer=LIMITE_IGAC["capa"],
)
igac = igac[
    igac[LIMITE_IGAC["columna_codigo"]]
    == LIMITE_IGAC["codigos_dane"][CIUDAD]
].to_crs(CRS_WGS84)

# --- Límite sustituido: IDECA ---
# Se lee de LIMITES_SUSTITUIDOS y no de cfg["limite"], porque
# cfg["limite"] ya apunta al IGAC: leerlo de ahí compararía la
# capa contra sí misma.
sust = LIMITES_SUSTITUIDOS[CIUDAD]
muni = gpd.read_file(
    sust["archivo"], layer=sust["capa"]
).to_crs(CRS_WGS84)

# --- bbox de consulta ---
b = cfg["bbox"]
bbox = gpd.GeoDataFrame(
    geometry=[box(b["oeste"], b["sur"], b["este"], b["norte"])],
    crs=CRS_WGS84,
)

# ---------------------------------------------------------------------------
# Mapa. Se construye con explore() y tiles=None, y el fondo se añade
# después como TileLayer explícito: es la forma que carga las teselas
# de manera confiable. explore() encuadra sobre la primera capa.
#
# Diferenciación cuando los tres bordes coinciden:
#   1. El IGAC va de primero con trazo ancho (9 px) y translúcido, de
#      modo que asoma como halo alrededor de las capas superiores.
#   2. muni y bbox comparten el patrón "10,10" pero con dashOffset
#      distinto, así que quedan intercalados: donde uno pinta, el
#      otro deja hueco.
#   3. lineCap "butt" evita que Leaflet redondee las puntas de cada
#      guion y rellene los huecos hasta volver la línea casi continua.
# ---------------------------------------------------------------------------

m = igac[["MpCodigo", "MpNombre", "Depto", "MpArea", "MpAltitud", "geometry"]].explore(
    color="#1D4ED8",
    style_kwds={
        "fillOpacity": 0.10,
        "weight": 9,
        "opacity": 0.55,
    },
    name="1 · IGAC (adoptado) — azul, halo ancho",
    tiles=None,
)

folium.TileLayer(
    "OpenStreetMap", name="Fondo: OpenStreetMap", overlay=False
).add_to(m)

folium.TileLayer(
    tiles=(
        "https://server.arcgisonline.com/ArcGIS/rest/services/"
        "World_Imagery/MapServer/tile/{z}/{y}/{x}"
    ),
    attr="Esri, Maxar, Earthstar Geographics",
    name="Fondo: Esri satelital",
    overlay=False,
).add_to(m)

muni.explore(
    m=m,
    color="#16A34A",
    style_kwds={
        "fill": False,
        "weight": 3.5,
        "opacity": 1.0,
        "dashArray": "10,10",
        "dashOffset": "0",
        "lineCap": "butt",
    },
    name="2 · IDECA (sustituido) — verde, guiones",
)

bbox.explore(
    m=m,
    color="#EA580C",
    style_kwds={
        "fill": False,
        "weight": 3.5,
        "opacity": 1.0,
        "dashArray": "10,10",
        "dashOffset": "10",
        "lineCap": "butt",
    },
    name="3 · bbox Overpass — naranja, guiones alternos",
)

folium.LayerControl(collapsed=False).add_to(m)
m

In [6]:
RUTA_MAPA = f"mapa_limites_{CIUDAD.lower()}.html"
m.save(RUTA_MAPA)
print(f"Mapa guardado en: {RUTA_MAPA}")

Mapa guardado en: mapa_limites_bogota.html


# Mapa limites Medellín

In [7]:
CIUDAD = "Medellin"
cfg = AREAS_ESTUDIO[CIUDAD]

# --- Capa adoptada: IGAC ---
igac_med = gpd.read_file(
    LIMITE_IGAC["archivo"],
    layer=LIMITE_IGAC["capa"],
)
igac_med = igac_med[
    igac_med[LIMITE_IGAC["columna_codigo"]]
    == LIMITE_IGAC["codigos_dane"][CIUDAD]
].to_crs(CRS_WGS84)

# --- Límite sustituido: derivado de OpenStreetMap vía Nominatim ---
# A diferencia de Bogotá, este límite NO era independiente de la
# fuente de estaciones: provenía de la misma base. Es la razón de
# fondo de la sustitución.
# Se lee de LIMITES_SUSTITUIDOS y no de cfg["limite"], porque
# cfg["limite"] ya apunta al IGAC: leerlo de ahí compararía la capa
# contra sí misma.
sust_med = LIMITES_SUSTITUIDOS[CIUDAD]
osm_med = gpd.read_file(
    sust_med["archivo"], layer=sust_med["capa"]
).to_crs(CRS_WGS84)

# --- bbox de consulta ---
b = cfg["bbox"]
bbox_med = gpd.GeoDataFrame(
    geometry=[box(b["oeste"], b["sur"], b["este"], b["norte"])],
    crs=CRS_WGS84,
)

# ---------------------------------------------------------------------------
# Mapa. Se construye con explore() y tiles=None, y el fondo se añade
# después como TileLayer explícito: es la forma que carga las teselas
# de manera confiable. explore() encuadra sobre la primera capa.
#
# Diferenciación cuando los tres bordes coinciden:
#   1. El IGAC va de primero con trazo ancho (9 px) y translúcido, de
#      modo que asoma como halo alrededor de las capas superiores.
#   2. El límite OSM y el bbox comparten el patrón "10,10" pero con
#      dashOffset distinto, así que quedan intercalados: donde uno
#      pinta, el otro deja hueco.
#   3. lineCap "butt" evita que Leaflet redondee las puntas de cada
#      guion y rellene los huecos hasta volver la línea casi continua.
# ---------------------------------------------------------------------------

m_med = igac_med[
    ["MpCodigo", "MpNombre", "Depto", "MpArea", "MpAltitud", "geometry"]
].explore(
    color="#1D4ED8",
    style_kwds={
        "fillOpacity": 0.10,
        "weight": 9,
        "opacity": 0.55,
    },
    name="1 · IGAC (adoptado) — azul, halo ancho",
    tiles=None,
)

folium.TileLayer(
    "OpenStreetMap", name="Fondo: OpenStreetMap", overlay=False
).add_to(m_med)

folium.TileLayer(
    tiles=(
        "https://server.arcgisonline.com/ArcGIS/rest/services/"
        "World_Imagery/MapServer/tile/{z}/{y}/{x}"
    ),
    attr="Esri, Maxar, Earthstar Geographics",
    name="Fondo: Esri satelital",
    overlay=False,
).add_to(m_med)

osm_med.explore(
    m=m_med,
    color="#16A34A",
    style_kwds={
        "fill": False,
        "weight": 3.5,
        "opacity": 1.0,
        "dashArray": "10,10",
        "dashOffset": "0",
        "lineCap": "butt",
    },
    name="2 · OSM/Nominatim (sustituido) — verde, guiones",
)

bbox_med.explore(
    m=m_med,
    color="#EA580C",
    style_kwds={
        "fill": False,
        "weight": 3.5,
        "opacity": 1.0,
        "dashArray": "10,10",
        "dashOffset": "10",
        "lineCap": "butt",
    },
    name="3 · bbox Overpass — naranja, guiones alternos",
)

folium.LayerControl(collapsed=False).add_to(m_med)
m_med

In [8]:
RUTA_MAPA_MED = f"mapa_limites_{CIUDAD.lower()}.html"
m_med.save(RUTA_MAPA_MED)
print(f"Mapa guardado en: {RUTA_MAPA_MED}")

Mapa guardado en: mapa_limites_medellin.html


# Comparación cuantitativa de las geometrías de delimitación

In [9]:
# Verificación cuantitativa del cambio de límite. Acompaña a los
# mapas anteriores: el mapa muestra dónde difieren las geometrías,
# estas métricas dicen cuánto.
#
# Todas las áreas se miden en EPSG:9377 (MAGNA-SIRGAS / Origen
# Nacional) y no en grados, porque un área calculada sobre
# coordenadas geográficas no tiene unidades interpretables.

CRS_METRICO = "EPSG:9377"


def area_km2(geom):
    return gpd.GeoSeries([geom], crs=CRS_WGS84).to_crs(CRS_METRICO).area.iloc[0] / 1e6


for clave in ["Bogota", "Medellin"]:
    cfg = AREAS_ESTUDIO[clave]

    # --- Límite adoptado: IGAC ---
    ig = gpd.read_file(
        LIMITE_IGAC["archivo"], layer=LIMITE_IGAC["capa"]
    )
    ig = ig[
        ig[LIMITE_IGAC["columna_codigo"]]
        == LIMITE_IGAC["codigos_dane"][clave]
    ].to_crs(CRS_WGS84).union_all()

    # --- Límite sustituido ---
    # Se lee de LIMITES_SUSTITUIDOS y no de cfg["limite"], porque
    # cfg["limite"] ya apunta al IGAC: leerlo de ahí compararía la
    # capa contra sí misma y daría IoU de 1,0.
    sust = LIMITES_SUSTITUIDOS[clave]
    an = gpd.read_file(
        sust["archivo"], layer=sust["capa"]
    ).to_crs(CRS_WGS84).union_all()

    # --- bbox de consulta ---
    b = cfg["bbox"]
    caja = box(b["oeste"], b["sur"], b["este"], b["norte"])

    print(f"=== {cfg['nombre']} ===")
    print(f"  area IGAC      : {area_km2(ig):.1f} km2")
    print(f"  area sustituido: {area_km2(an):.1f} km2")
    print(f"  IoU sust vs igac: {area_km2(ig.intersection(an)) / area_km2(ig.union(an)):.4f}")
    print(f"  solo sustituido: {area_km2(an.difference(ig)):.1f} km2")
    print(f"  solo IGAC      : {area_km2(ig.difference(an)):.1f} km2")
    print(f"  bbox fuera IGAC: {area_km2(caja.difference(ig)):.1f} km2")
    print(f"  IGAC fuera bbox: {area_km2(ig.difference(caja)):.1f} km2")
    print(f"  motivo         : {sust['motivo_sustitucion'][:60]}...")

=== Bogotá D.C. ===
  area IGAC      : 1618.2 km2
  area sustituido: 1633.1 km2
  IoU sust vs igac: 0.9763
  solo sustituido: 27.0 km2
  solo IGAC      : 12.0 km2
  bbox fuera IGAC: 338.2 km2
  IGAC fuera bbox: 1010.2 km2
  motivo         : Trazabilidad incompleta: la ficha exacta del recurso no qued...
=== Medellín ===
  area IGAC      : 373.4 km2
  area sustituido: 374.6 km2
  IoU sust vs igac: 0.9835
  solo sustituido: 3.7 km2
  solo IGAC      : 2.5 km2
  bbox fuera IGAC: 268.6 km2
  IGAC fuera bbox: 0.0 km2
  motivo         : Dependencia circular: el límite provenía de OpenStreetMap, l...


# 8. Construcción de la consulta Overpass

Cerrada la validación de la configuración, comienza la ejecución del pipeline. Este bloque define dos funciones: una que arma el texto de la consulta enviada a Overpass API y otra que calcula su huella criptográfica.

## 8.1. construir_consulta_overpass

La consulta no se escribe como texto fijo sino que se genera a partir de los parámetros declarados. Esto garantiza que el texto efectivamente enviado corresponda siempre a la configuración vigente, sin posibilidad de que ambos se desincronicen.

La función recibe el bbox del área de estudio, una fecha de corte opcional y un tiempo máximo de resolución que toma por defecto el valor de TIMEOUT_OVERPASS_QUERY.

Validación previa. Antes de construir nada, la función invoca validar_bbox. La validación se repite aquí aunque ya se haya ejecutado sobre AREAS_ESTUDIO, porque la función puede recibir un bbox construido en otro punto del cuaderno. Una función que produce el insumo de una petición externa no debe confiar en que alguien más validó sus argumentos.

Bloque de ajustes. Se compone el encabezado de la consulta con el formato de salida JSON y el tiempo máximo de resolución interpolado desde la constante. El valor no se escribe a mano: si la constante cambia, la consulta cambia con ella.

Conviene declarar el alcance de esa generación. El tiempo máximo, el bbox y la fecha de corte se interpolan desde los parámetros; la etiqueta `amenity=charging_station`, en cambio, está escrita literalmente en el cuerpo de la consulta y no se toma de `ETIQUETA_OSM`, que solo se emplea en los metadatos. Modificar esa constante no modificaría la consulta. Es una duplicación consciente y de bajo riesgo —el criterio de selección del entregable no está previsto que cambie— pero no debe leerse como si el texto se derivara íntegramente de la configuración.

Cuando se entrega una fecha de corte, se añade la directiva [date:"..."], que instruye a Overpass a devolver el estado del mapa en ese instante en lugar del estado actual. Es la attic query descrita en la sección 5.6, y es lo que distingue una descarga reproducible de una meramente re-ejecutable: sin ella, dos corridas del mismo cuaderno en fechas distintas pueden devolver inventarios distintos sin que nada lo advierta.

Orden de coordenadas. El bbox se serializa en el orden que Overpass exige: sur, oeste, norte, este. Este orden no coincide con el que emplean la mayoría de las bibliotecas geoespaciales, que suelen usar (oeste, sur, este, norte). La conversión se hace explícitamente en este punto, y el diccionario conserva sus claves con nombre precisamente para que la correspondencia sea legible y no dependa de recordar una convención posicional.

Cuerpo de la consulta. Se solicitan los tres tipos de elemento que pueden representar una estación en OpenStreetMap (node, way y relation) todos filtrados por la etiqueta amenity=charging_station y acotados al mismo rectángulo. Omitir cualquiera de los tres produciría una pérdida silenciosa: una estación mapeada como polígono no aparece en una consulta restringida a nodos.

Instrucción de salida. La consulta termina con out meta center, y ambos modificadores cumplen una función.

center hace que las geometrías extensas (vías y relaciones) se devuelvan reducidas a un punto representativo, lo que las vuelve homogéneas con los nodos y comparables con los registros puntuales de OpenChargeMap.

meta añade a cada elemento su versión, marca temporal, conjunto de cambios y usuario. Esta información no es accesoria. La marca temporal de cada elemento es una señal de vigencia comparable a la fecha de última verificación de OpenChargeMap, y el historial de versiones es parte de la justificación por la cual OpenStreetMap se adoptó como fuente: sus registros son auditables individualmente, cosa que la fuente descartada no permitía.

## 8.2. hash_consulta

Calcula el resumen SHA-256 del texto de la consulta.

Su propósito es probatorio. Permite afirmar en el informe que dos ejecuciones enviaron exactamente el mismo texto, sin necesidad de transcribir la consulta completa ni de que el lector la compare carácter por carácter. El hash se registra junto con los demás metadatos de la descarga.

In [10]:
def construir_consulta_overpass(
    bbox,
    fecha_corte=None,
    timeout=TIMEOUT_OVERPASS_QUERY,
):
    """
    Construye la consulta Overpass para estaciones de carga.

    Cambios respecto de la versión anterior:

    1. El timeout se interpola desde la constante,
       en lugar de estar escrito a mano.

    2. Si se entrega fecha_corte, se añade [date:"..."],
       que devuelve el estado histórico del mapa en ese
       instante y vuelve la descarga reproducible y no
       solo re-ejecutable.

    3. 'out meta center' en lugar de 'out center tags':
       meta añade version, timestamp, changeset y user
       por elemento. El timestamp es una señal de vigencia
       y el historial de versiones es parte de la
       justificación de OSM como fuente.
    """

    validar_bbox(
        bbox,
        "área de estudio"
    )

    ajustes = (
        f"[out:json]"
        f"[timeout:{timeout}]"
    )

    if fecha_corte:

        ajustes += (
            f'[date:"{fecha_corte}"]'
        )

    caja = (
        f'{bbox["sur"]},'
        f'{bbox["oeste"]},'
        f'{bbox["norte"]},'
        f'{bbox["este"]}'
    )

    query = f"""
{ajustes};

(
    node["amenity"="charging_station"]({caja});
    way["amenity"="charging_station"]({caja});
    relation["amenity"="charging_station"]({caja});
);

out meta center;
""".strip()

    return query


def hash_consulta(query):
    """
    Huella de la consulta, para poder afirmar en el informe
    que dos corridas ejecutaron exactamente el mismo texto.
    """

    return hashlib.sha256(
        query.encode("utf-8")
    ).hexdigest()

# 9. Validación de la consulta Overpass

Construida la consulta, se verifica que efectivamente contenga los elementos que el entregable exige antes de enviarla al servidor. La función validar_consulta_overpass realiza esa comprobación sobre el texto ya generado.

La validación opera sobre el producto y no sobre los parámetros que lo originaron. Esta distinción es lo que le da sentido: construir_consulta_overpass puede modificarse, reformatearse o adquirir nuevas ramas condicionales, y esta función seguirá comprobando el resultado con independencia de cómo se haya obtenido. Es una verificación por un camino distinto al que produjo el objeto, que es el criterio de trabajo adoptado en el proyecto tras el hallazgo del error de emparejamiento del Bloque A.

## 9.1. Estructura de la comprobación

Las verificaciones se organizan en un diccionario que asocia el nombre legible de cada requisito con el resultado booleano de comprobarlo. Los requisitos se evalúan todos, y solo después se recogen los que fallaron.

Esta organización es deliberada y difiere de una cadena de condicionales que se detenga en el primer fallo. Al evaluar el conjunto completo, el mensaje de error enumera todas las deficiencias simultáneamente, lo que permite corregirlas en una sola pasada en lugar de descubrirlas una por una en ejecuciones sucesivas.

Al igual que validar_bbox, la función interrumpe la ejecución mediante ValueError cuando algún requisito no se cumple, y devuelve True cuando todos pasan. Una consulta incompleta no admite continuación razonable: enviarla produciría una descarga silenciosamente parcial, que es precisamente el tipo de fallo que el proyecto busca evitar.

## 9.2. Requisitos verificados

Criterio de selección. Se comprueba que la etiqueta amenity=charging_station esté presente en la consulta. Es el criterio de inclusión que define el inventario, y su ausencia devolvería un conjunto de elementos ajeno al objeto del trabajo.

Cobertura de los tres tipos de elemento. Se verifica por separado que la etiqueta aparezca aplicada a node, a way y a relation. La comprobación no se limita a constatar que las tres palabras figuren en el texto: exige que cada una aparezca acompañada del filtro por etiqueta.

Este requisito merece atención porque protege contra una pérdida que no se manifestaría como error. Una estación mapeada en OpenStreetMap como polígono (caso frecuente cuando se representa el área de un electrolinera completa—)no aparece en una consulta restringida a nodos. La descarga se completaría con éxito, devolvería un archivo válido y arrojaría un conteo plausible, solo que incompleto. Verificar los tres tipos convierte esa omisión potencial en un fallo inmediato.

Instrucción de salida. Se comprueba la presencia de out meta center, que gobierna qué información devuelve el servidor por cada elemento. Si se degradara a una forma más simple, se perderían tanto el punto representativo de las geometrías extensas como los metadatos de versión y marca temporal descritos en la sección 8.1.

Formato y tiempo máximo. Se verifica que la consulta declare salida en JSON y que incluya una directiva de tiempo máximo de resolución. La ausencia del formato produciría una respuesta que el código posterior no sabría interpretar; la ausencia del tiempo máximo dejaría la consulta sujeta al valor por defecto del servidor, que varía entre espejos y sustraería el parámetro al control del cuaderno.

Corte temporal, cuando aplica. Este requisito se añade condicionalmente: solo se comprueba si se entregó una fecha de corte. La verificación no se limita a constatar que exista una directiva [date:...], sino que confirma que la fecha presente en la consulta sea exactamente la solicitada.


## 9.3. Normalización a minúsculas

La comprobación se realiza sobre una versión del texto convertida íntegramente a minúsculas, y los patrones buscados están escritos en ese mismo registro. La normalización evita que una diferencia de capitalización haga fallar una consulta por lo demás correcta.

Cabe señalar una consecuencia de esta decisión: al normalizar también la fecha de corte antes de compararla, la validación acepta indistintamente el sufijo de zona horaria en mayúscula o minúscula. Overpass admite ambas formas, de modo que la tolerancia es correcta, pero conviene tenerla presente al interpretar qué se está verificando exactamente: que la consulta declara una fecha de corte equivalente a la solicitada, no que la declara con la escritura exacta que se entregó.

In [11]:
def validar_consulta_overpass(
    query,
    fecha_corte=None,
):
    """
    Verifica que la consulta contenga los elementos
    requeridos para el entregable A2.
    """

    consulta = query.lower()

    requisitos = {

        "amenity=charging_station":
            '"amenity"="charging_station"' in consulta,

        "node":
            'node["amenity"="charging_station"]' in consulta,

        "way":
            'way["amenity"="charging_station"]' in consulta,

        "relation":
            'relation["amenity"="charging_station"]' in consulta,

        "out meta center":
            "out meta center" in consulta,

        "formato json":
            "[out:json]" in consulta,

        "timeout declarado":
            "[timeout:" in consulta,
    }

    if fecha_corte:

        requisitos["corte temporal declarado"] = (
            f'[date:"{fecha_corte.lower()}"]'
            in consulta
        )

    errores = [
        requisito
        for requisito, resultado in requisitos.items()
        if not resultado
    ]

    if errores:

        raise ValueError(
            "La consulta Overpass no cumple los requisitos: "
            f"{errores}"
        )

    return True

In [12]:
# Verificación independiente: los números del bbox declarados en
# AREAS_ESTUDIO deben aparecer escritos en el texto de la consulta,
# una vez por cada tipo de elemento (node, way, relation).

def verificar_bbox_en_consulta(query, bbox, nombre_ciudad):

    caja = (
        f'{bbox["sur"]},'
        f'{bbox["oeste"]},'
        f'{bbox["norte"]},'
        f'{bbox["este"]}'
    )

    apariciones = query.count(caja)

    if apariciones != 3:
        raise ValueError(
            f"{nombre_ciudad}: el bbox declarado aparece "
            f"{apariciones} veces en la consulta; se esperaban 3 "
            f"(node, way, relation). Cadena buscada: {caja}"
        )

    print(f"{nombre_ciudad}: bbox correcto en la consulta ({caja})")
    return True


# --- Ejecución sobre las dos ciudades ---
for clave, ciudad in AREAS_ESTUDIO.items():

    consulta_ciudad = construir_consulta_overpass(
        ciudad["bbox"],
        fecha_corte=FECHA_CORTE_OSM,
    )

    verificar_bbox_en_consulta(
        consulta_ciudad,
        ciudad["bbox"],
        ciudad["nombre"],
    )

Bogotá D.C.: bbox correcto en la consulta (4.4711,-74.2235,4.8331,-74.0102)
Medellín: bbox correcto en la consulta (6.1629,-75.7193,6.3748,-75.4718)


# 10. Descarga mediante Overpass API

Validada la consulta, se ejecuta. La función descargar_estaciones_osm gestiona el envío, los servidores de contingencia, los reintentos y el registro de todo lo ocurrido durante el proceso.

La función no se limita a obtener los datos: devuelve un diccionario que incluye el resultado junto con la trazabilidad completa de cómo se obtuvo. Esa combinación es deliberada. Separar los datos de las circunstancias de su descarga permitiría que ambos se desincronizaran; entregarlos juntos garantiza que todo archivo producido conserve la evidencia de su origen.

## 10.1. Estrategia de contingencia

La ejecución recorre los servidores declarados en OVERPASS_SERVERS en orden, y sobre cada uno realiza hasta MAX_REINTENTOS intentos separados por ESPERA_REINTENTO segundos. La primera respuesta que supere todas las comprobaciones se acepta y la función retorna de inmediato.

La estructura anidada distingue dos clases de fallo. Los reintentos sobre un mismo servidor atienden fallos transitorios (saturación momentánea, corte de red) que suelen resolverse esperando. El paso al siguiente servidor atiende fallos persistentes: un espejo caído o que rechaza sistemáticamente la consulta.

Si se agotan todos los servidores sin obtener una respuesta válida, la función interrumpe la ejecución mediante RuntimeError e incluye el último error registrado. No devuelve un resultado parcial ni vacío: una descarga fallida debe manifestarse como tal.

## 10.2. El fallo silencioso de Overpass

La comprobación más importante de esta función responde a un comportamiento particular del servicio que la versión anterior del código no contemplaba.

Cuando una consulta agota el tiempo de resolución o la memoria disponible del servidor, Overpass no responde con un código de error HTTP. Responde con un código 200 (éxito) y un cuerpo que contiene una lista de elementos vacía junto a un campo remark describiendo el problema.

Para un cliente que solo verifique el código de estado, esa respuesta es indistinguible de una consulta legítima sobre una zona sin estaciones. La versión anterior la daba por exitosa, retornaba con cero elementos y nunca llegaba a probar los servidores de contingencia. El fallo se manifestaba como un inventario vacío sin explicación aparente.

La corrección consiste en inspeccionar el contenido de la respuesta y no solo su código. La presencia del campo remark se trata como fallo y desencadena el mecanismo de contingencia, con el texto del servidor incorporado al mensaje de error para que la causa quede registrada.

## 10.3. Comprobaciones sobre la respuesta

Superada la verificación de remark, la respuesta atraviesa dos comprobaciones adicionales antes de aceptarse.

Estructura. Se verifica que el campo elements sea efectivamente una lista. Una respuesta malformada, o el resultado de un error de comunicación parcial, podría llegar como un tipo distinto y provocar fallos posteriores en el procesamiento, lejos del punto de origen.

Contenido no vacío. Una lista sin elementos se trata como fallo y se pasa al siguiente servidor.

Esta última decisión merece explicitarse como supuesto y no como regla general. Se apoya en el conocimiento previo de que ambas ciudades de estudio cuentan con estaciones registradas en OpenStreetMap, de modo que un resultado vacío indica con alta probabilidad un problema del servidor y no una ausencia real. El comportamiento es adecuado para las áreas de este trabajo, pero la función no distinguiría, sobre un área efectivamente sin estaciones, entre esa ausencia legítima y un fallo del servicio: agotaría los cuatro servidores y terminaría en error. Aplicar esta función a otras ciudades exigiría revisar el supuesto.

## 10.4. La marca temporal que importa

Cuando la descarga tiene éxito, la función conserva el bloque osm3s que Overpass incluye en su respuesta, y en particular el campo timestamp_osm_base.

La distinción respecto de las marcas temporales locales es esencial. fecha_inicio y fecha_fin registran cuándo se ejecutó el cuaderno, según el reloj de la máquina. timestamp_osm_base registra hasta qué momento estaban actualizados los datos del servidor cuando respondió.

Se conservan también timestamp_areas_base y el identificador del generator, que documentan la versión del servicio que atendió la consulta.

## 10.5. Registro de los intentos

La función construye una bitácora de cada intento realizado, con el servidor consultado, el número de intento y su resultado exitoso, o el texto del error que lo hizo fallar. Esa bitácora viaja en el diccionario de retorno.

El registro no se limita al intento exitoso. Si el primer servidor falló dos veces y el segundo respondió, los tres eventos quedan consignados. Esto permite reportar en el informe no solo de dónde provienen los datos, sino qué ocurrió para que provinieran de ahí, y hace visible la degradación de los servicios de contingencia a lo largo del tiempo.

## 10.5b. El congelamiento del archivo crudo

Fijar el corte hace la descarga reproducible en su resultado, pero deja el resultado dependiendo de que el servicio siga prestando datos históricos y respondiendo de la misma manera. El congelamiento cierra esa dependencia.

La primera ejecución escribe tres archivos: el crudo, con los bytes exactos que devolvió Overpass; su huella SHA-256; y un archivo de metadatos con las circunstancias de la descarga, que el crudo no contiene. El nombre del crudo lleva la fecha del corte y no la del día de ejecución, porque lo que identifica al archivo es el estado del mapa que describe.

Toda ejecución posterior lee de ese archivo en lugar de consultar el servidor. Antes de usarlo comprueba tres condiciones: que la huella coincida con la registrada, que se haya descargado con exactamente la misma consulta, y que corresponda al corte vigente. Un crudo que falle cualquiera de las tres no se reutiliza y la ejecución se detiene.

Los bytes se escriben tal como llegaron, sin reserializar. La distinción no es menor: una huella calculada sobre el resultado de volver a imprimir el JSON desde Python no certificaría lo que el servidor envió, sino lo que Python decidió escribir.

Redescargar exige borrar los tres archivos. Es deliberado: debe ser una acción consciente y no algo que pueda ocurrir por reejecutar una celda.

## 10.6. Contenido del resultado

El diccionario devuelto reúne cuatro clases de información: los elementos descargados; la identificación del servidor que respondió y la bitácora de intentos; las marcas temporales, tanto locales como las declaradas por el servidor; y la fecha de corte solicitada, que permite contrastar lo pedido contra lo obtenido.

In [13]:
def rutas_crudo(ciudad, fecha_corte_local=FECHA_CORTE_LOCAL):
    """
    Nombres de los tres archivos del congelamiento.

    El crudo guarda los bytes exactos que devolvió Overpass; el
    .sha256 permite comprobar que no se modificaron; el .meta.json
    conserva las circunstancias de la descarga, que el crudo no
    contiene.
    """

    base = f"{PREFIJO_CRUDO}_{ciudad}_{fecha_corte_local}"

    return {
        "crudo": f"{base}.json",
        "hash": f"{base}.sha256",
        "meta": f"{base}.meta.json",
    }


def sha256_archivo(ruta):
    """Huella SHA-256 del contenido de un archivo en disco."""

    with open(ruta, "rb") as archivo:
        return hashlib.sha256(archivo.read()).hexdigest()


def verificar_corte_respetado(elementos, fecha_corte, ciudad):
    """
    Comprueba que el servidor honró la attic query.

    La verificación NO puede hacerse contra timestamp_osm_base: ese
    campo declara hasta qué momento está actualizada la base de datos
    del servidor y devuelve la fecha de hoy aunque el corte se haya
    respetado íntegramente. Es el estado del servidor, no el de la
    consulta.

    La evidencia real está en los datos. Como la consulta lleva
    'out meta', cada elemento trae la fecha de su última edición. Si
    el corte se respetó, ninguna puede ser posterior al instante
    solicitado; si el servidor ignoró la directiva, aparecerán
    elementos editados después.

    La prueba tiene una propiedad deseable: cuando ningún elemento fue
    editado entre el corte y el presente, pasa en ambos casos, y con
    razón, porque entonces el estado histórico y el vigente son el
    mismo conjunto de datos.

    Devuelve la fecha de edición más reciente hallada, que se registra
    en los metadatos como constancia.
    """

    if not fecha_corte:
        return None

    solicitado = datetime.fromisoformat(
        fecha_corte.replace("Z", "+00:00")
    )

    marcas = []

    for elemento in elementos:

        marca = elemento.get("timestamp")

        if marca:
            marcas.append(marca)

    if not marcas:

        raise RuntimeError(
            f"{ciudad}: ningún elemento trae marca temporal. La "
            "consulta debe incluir 'out meta' para poder verificar "
            "el corte; sin ella la attic query no es comprobable."
        )

    fechas = [
        datetime.fromisoformat(m.replace("Z", "+00:00"))
        for m in marcas
    ]

    posteriores = [
        f for f in fechas if f > solicitado
    ]

    if posteriores:

        raise RuntimeError(
            f"{ciudad}: se pidió el estado del mapa al {fecha_corte} "
            f"y {len(posteriores)} de {len(fechas)} elementos fueron "
            "editados después de ese instante. El más reciente es del "
            f"{max(posteriores).isoformat()}. El espejo ignoró la "
            "attic query: lo descargado no es el corte solicitado."
        )

    mas_reciente = max(fechas).isoformat()

    print(
        f"  Corte verificado: edición más reciente {mas_reciente}, "
        f"anterior al corte solicitado ({len(fechas)} elementos)."
    )

    return mas_reciente


def leer_crudo_congelado(rutas, query, fecha_corte, ciudad):
    """
    Reconstruye el resultado de la descarga a partir del archivo
    crudo ya congelado, en lugar de volver a consultar Overpass.

    Antes de usarlo comprueba tres cosas: que el archivo no se haya
    modificado (hash), que se haya descargado con exactamente esta
    consulta, y que corresponda al corte temporal vigente. Un crudo
    que no supere las tres no es reutilizable.
    """

    for clave in ("hash", "meta"):

        if not os.path.exists(rutas[clave]):
            raise RuntimeError(
                f"{ciudad}: existe {rutas['crudo']} pero falta "
                f"{rutas[clave]}. Un crudo sin su huella no se puede "
                "verificar. Borre los tres archivos y vuelva a descargar."
            )

    with open(rutas["hash"], encoding="utf-8") as archivo:
        hash_declarado = archivo.read().split()[0].strip()

    hash_real = sha256_archivo(rutas["crudo"])

    if hash_real != hash_declarado:
        raise RuntimeError(
            f"{ciudad}: el archivo crudo fue modificado después de "
            f"congelarse. SHA-256 declarado {hash_declarado}, "
            f"recalculado {hash_real}."
        )

    with open(rutas["meta"], encoding="utf-8") as archivo:
        meta = json.load(archivo)

    if meta.get("hash_consulta") != hash_consulta(query):
        raise RuntimeError(
            f"{ciudad}: el crudo se descargó con una consulta distinta "
            "de la que el cuaderno construye ahora. No es reutilizable: "
            "restituya la configuración original o borre los tres "
            "archivos para redescargar."
        )

    if meta.get("fecha_corte_solicitada") != fecha_corte:
        raise RuntimeError(
            f"{ciudad}: el crudo corresponde al corte "
            f"{meta.get('fecha_corte_solicitada')!r} y la configuración "
            f"vigente pide {fecha_corte!r}."
        )

    with open(rutas["crudo"], encoding="utf-8") as archivo:
        data = json.load(archivo)

    osm3s = data.get("osm3s", {})
    elementos = data.get("elements", [])

    verificar_corte_respetado(
        elementos,
        fecha_corte,
        ciudad,
    )

    print(f"Archivo congelado: {rutas['crudo']}")
    print(f"  SHA-256 verificado: {hash_real[:16]}")
    print(
        f"  {len(elementos)} elementos | "
        f"corte OSM {osm3s.get('timestamp_osm_base')}"
    )

    return {
        "elementos": elementos,
        "origen": "archivo crudo congelado",
        "servidor": meta.get("servidor"),
        "fecha_inicio": meta.get("fecha_inicio"),
        "fecha_fin": meta.get("fecha_fin"),
        "duracion_segundos": meta.get("duracion_segundos"),
        "timestamp_osm_base": osm3s.get("timestamp_osm_base"),
        "timestamp_areas_base": osm3s.get("timestamp_areas_base"),
        "generador": data.get("generator"),
        "fecha_corte_solicitada": meta.get("fecha_corte_solicitada"),
        "servidores_intentados": [],
        "ruta_crudo": rutas["crudo"],
        "hash_crudo": hash_real,
    }


def descargar_estaciones_osm(query, ciudad, fecha_corte=None):
    """
    Obtiene los elementos de OpenStreetMap para una ciudad.

    Tiene dos vías y siempre prefiere la primera:

    1. Si el archivo crudo del corte ya existe en disco, se lee de
       ahí previa verificación de su huella. Ninguna ejecución
       posterior vuelve a consultar el servidor.

    2. Si no existe, se descarga y se congela: se escriben los bytes
       exactos de la respuesta, su SHA-256 y un archivo de metadatos.

    Es la disciplina exigida por la tutoría: el crudo se guarda con
    la fecha en el nombre y su hash, y todo lo demás sale de ahí.
    OpenChargeMap no admite consultas históricas, de modo que su
    archivo debe congelarse por fuerza; congelar también el de OSM
    mantiene las dos fuentes en el mismo instante.
    """

    rutas = rutas_crudo(ciudad)

    if os.path.exists(rutas["crudo"]):
        return leer_crudo_congelado(
            rutas,
            query,
            fecha_corte,
            ciudad,
        )

    # ------------------------------------------------------------------------
    # No hay crudo: se descarga y se congela
    # ------------------------------------------------------------------------
    # Con corte fijado solo se consultan los espejos con histórico.

    servidores = (
        OVERPASS_SERVERS_HISTORICOS
        if fecha_corte
        else OVERPASS_SERVERS
    )

    fecha_inicio = datetime.now(TIMEZONE)

    headers = {"User-Agent": USER_AGENT}

    ultimo_error = None

    servidores_intentados = []

    for servidor in servidores:

        for intento in range(1, MAX_REINTENTOS + 1):

            print(f"Consultando: {servidor}")
            print(f"Intento {intento}/{MAX_REINTENTOS}")

            registro_intento = {
                "servidor": servidor,
                "intento": intento,
                "resultado": None,
            }

            servidores_intentados.append(registro_intento)

            try:

                response = requests.post(
                    servidor,
                    data={"data": query},
                    headers=headers,
                    timeout=TIMEOUT_OVERPASS_HTTP,
                )

                response.raise_for_status()

                data = response.json()

                # Fallo silencioso: HTTP 200 con remark.

                remark = data.get("remark")

                if remark:
                    raise RuntimeError(
                        "Overpass respondió con remark "
                        f"(fallo del servidor): {remark}"
                    )

                elementos = data.get("elements")

                if not isinstance(elementos, list):
                    raise ValueError(
                        "La respuesta de Overpass no contiene una "
                        "lista válida de elementos."
                    )

                if len(elementos) == 0:
                    raise RuntimeError(
                        "Overpass devolvió cero elementos. Se descarta "
                        "la respuesta y se intenta otro servidor."
                    )

                osm3s = data.get("osm3s", {})

                # Segundo fallo silencioso: espejo que ignora [date:].
                # Se trata como fallo del servidor, de modo que se
                # pruebe el siguiente en lugar de aceptar un corte
                # que no es el pedido.

                edicion_mas_reciente = verificar_corte_respetado(
                    elementos,
                    fecha_corte,
                    ciudad,
                )

                fecha_fin = datetime.now(TIMEZONE)

                # ---------------------------------------------------
                # Congelamiento
                # ---------------------------------------------------
                # Se escriben los bytes tal como llegaron, sin
                # reserializar: el hash debe corresponder a lo que
                # el servidor envió y no a lo que Python decidió
                # imprimir.

                with open(rutas["crudo"], "wb") as archivo:
                    archivo.write(response.content)

                hash_crudo = sha256_archivo(rutas["crudo"])

                with open(
                    rutas["hash"], "w", encoding="utf-8"
                ) as archivo:
                    archivo.write(
                        f"{hash_crudo}  {rutas['crudo']}\n"
                    )

                meta = {
                    "ciudad": ciudad,
                    "servidor": servidor,
                    "generador": data.get("generator"),
                    "fecha_inicio": fecha_inicio.isoformat(),
                    "fecha_fin": fecha_fin.isoformat(),
                    "duracion_segundos": (
                        fecha_fin - fecha_inicio
                    ).total_seconds(),
                    "fecha_corte_solicitada": fecha_corte,
                    "fecha_corte_local": FECHA_CORTE_LOCAL,
                    "timestamp_osm_base": osm3s.get(
                        "timestamp_osm_base"
                    ),
                    "timestamp_areas_base": osm3s.get(
                        "timestamp_areas_base"
                    ),
                    "hash_consulta": hash_consulta(query),
                    "edicion_mas_reciente": edicion_mas_reciente,
                }

                with open(
                    rutas["meta"], "w", encoding="utf-8"
                ) as archivo:
                    json.dump(
                        meta,
                        archivo,
                        ensure_ascii=False,
                        indent=2,
                    )

                registro_intento["resultado"] = "exitoso"

                print(f"✓ Descarga exitosa: {len(elementos)} elementos.")
                print(
                    "  Corte de datos del servidor: "
                    f"{osm3s.get('timestamp_osm_base')}"
                )
                print(f"  Crudo congelado en: {rutas['crudo']}")
                print(f"  SHA-256: {hash_crudo[:16]}")

                return {
                    "elementos": elementos,
                    "origen": "descarga nueva, congelada en disco",
                    "servidor": servidor,
                    "fecha_inicio": fecha_inicio.isoformat(),
                    "fecha_fin": fecha_fin.isoformat(),
                    "duracion_segundos": (
                        fecha_fin - fecha_inicio
                    ).total_seconds(),
                    "timestamp_osm_base": osm3s.get(
                        "timestamp_osm_base"
                    ),
                    "timestamp_areas_base": osm3s.get(
                        "timestamp_areas_base"
                    ),
                    "generador": data.get("generator"),
                    "fecha_corte_solicitada": fecha_corte,
                    "servidores_intentados": servidores_intentados,
                    "ruta_crudo": rutas["crudo"],
                    "hash_crudo": hash_crudo,
                }

            except Exception as error:

                print(f"✗ Error: {error}")

                registro_intento["resultado"] = f"fallo: {error}"

                ultimo_error = error

                time.sleep(ESPERA_REINTENTO)

    raise RuntimeError(
        "No fue posible ejecutar la consulta en ninguno de los "
        f"servidores disponibles. Último error: {ultimo_error}"
    )

# 11. Transformación de los elementos OSM

Obtenida la respuesta de Overpass, transformar_elementos_osm convierte la estructura anidada que devuelve el servicio en un GeoDataFrame de columnas planas, apto para exportarse y para alimentar las etapas siguientes.

La función devuelve dos objetos: el GeoDataFrame y un diccionario de auditoría con el balance del proceso. Esa segunda salida es tan relevante como la primera, y su motivo se detalla en la sección 11.6.

## 11.1. Homogeneización de las coordenadas

Los tres tipos de elemento de OpenStreetMap no entregan su posición de la misma manera. Un node trae las coordenadas directamente en los campos lat y lon. Un way o una relation no tienen coordenadas propias: son geometrías extensas, y lo que Overpass devuelve —gracias al modificador center de la consulta— es un punto representativo dentro de un subobjeto center.

La función atiende primero el caso directo y, si no encuentra coordenadas, recurre al subobjeto. El resultado es que los tres tipos quedan representados como puntos, homogéneos entre sí y comparables con los registros puntuales de OpenChargeMap.

Conviene declarar el costo de esa homogeneización. Un way que representa el área de una electrolinera se reduce a un punto, y la extensión real de la instalación se pierde. Para el propósito de este trabajo (determinar dónde puede cargarse un vehículo) la simplificación es aceptable, pero implica que la distancia entre un registro de OpenStreetMap y uno de OpenChargeMap puede reflejar en parte esa reducción y no un desacuerdo posicional genuino.

Los elementos que carecen de coordenadas por ambas vías se descartan y se contabilizan por separado en la auditoría.

## 11.2. Conservación íntegra de las etiquetas

El campo tags_json almacena el diccionario completo de etiquetas de cada elemento, serializado como cadena de texto.

Esta decisión corrige un comportamiento de la versión anterior, que seleccionaba nueve etiquetas de interés y descartaba el resto. El problema de esa aproximación es que la selección se hace en el momento de la descarga, cuando todavía no se sabe qué etiquetas resultarán necesarias más adelante. Requerir cualquier otra obligaría a repetir la descarga, y una descarga repetida en fecha distinta ya no devuelve el mismo estado del mapa.

Conservar el diccionario íntegro desacopla la decisión de qué usar de la decisión de qué descargar. Las columnas planas siguen existiendo para las etiquetas de uso frecuente, pero el respaldo completo permanece disponible sin costo adicional de red.

El campo se acompaña de n_tags, que registra cuántas etiquetas trae cada elemento y permite cuantificar la riqueza de la descripción sin necesidad de deserializar.

## 11.3. Aislamiento de los conectores

El campo sockets_json recoge el subconjunto de etiquetas cuya clave comienza por socket:, y n_sockets_declarados cuenta cuántas hay.

Este subconjunto tiene una función precisa: es el equivalente en OpenStreetMap del tipo de conector y la potencia por conector que OpenChargeMap declara en campos propios. Sin él, la reconciliación de atributos que exige el Bloque B no puede realizarse, porque las dos fuentes describirían la misma característica en formatos incomparables.

Aislarlo en un campo propio, además de conservarlo dentro de tags_json, permite medir directamente cuántas estaciones declaran conectores sin recorrer el diccionario completo.

## 11.4. Campos de vigencia y metadatos

Se extraen dos etiquetas relacionadas con la vigencia del registro: `check_date` y `survey:date`. La primera es el análogo directo de la fecha de última verificación de OpenChargeMap.

Conviene registrar el resultado obtenido: **ninguna de las estaciones descargadas declara `check_date`**, ni en Bogotá ni en Medellín. La etiqueta existe en el esquema de OpenStreetMap pero la comunidad local no la emplea en estos registros. La consecuencia es metodológica y afecta al análisis de vigencia previsto en B5: no puede apoyarse en `check_date` para el lado de OpenStreetMap, porque el campo está vacío en la totalidad del inventario.

La alternativa disponible es `osm_timestamp`, la marca temporal de la última edición del elemento, que el modificador `meta` de la consulta hace disponible y que sí está presente en todos los registros. No es equivalente: `check_date` declara que alguien verificó la información en terreno, mientras que `osm_timestamp` solo indica cuándo se editó por última vez el registro, sea cual sea el motivo. La sustitución debe declararse como tal al reportar el análisis de vigencia, y no presentarse como si ambas fuentes se estuvieran tratando con el mismo criterio.

Se conservan asimismo los metadatos que el modificador `meta` de la consulta hizo disponibles: versión del elemento, marca temporal, conjunto de cambios y usuario. Estos campos sostienen la afirmación de que los registros de OpenStreetMap son auditables individualmente, que fue parte de la justificación para adoptar esta fuente.

## 11.5. Serialización de los campos anidados

Tanto tags_json como sockets_json se almacenan como cadenas de texto y no como diccionarios.

La razón es una restricción del formato de salida: el controlador GeoJSON no admite objetos anidados dentro de las propiedades de una entidad. Guardarlos como diccionario produciría un fallo en la exportación, o una conversión implícita cuyo resultado no está garantizado.

tags_json se serializa además con las claves ordenadas. Esto hace que dos elementos con el mismo conjunto de etiquetas produzcan siempre la misma cadena, lo que vuelve el campo comparable directamente y estable entre ejecuciones.

La consecuencia práctica es que cualquier uso posterior de estos campos requiere deserializarlos explícitamente.

## 11.6. La auditoría del proceso

El segundo objeto devuelto por la función es un diccionario que contabiliza qué entró, qué salió y qué se perdió por el camino: elementos recibidos, registros efectivamente transformados, descartados por falta de coordenadas y descartados por tipo no reconocido.

El propósito es que la transformación tenga un balance verificable. Si la suma de los registros producidos y los descartados no coincide con los elementos recibidos, hay una pérdida que el código no está contabilizando. Sin este balance, una pérdida silenciosa se manifestaría más adelante como un conteo menor al esperado, sin indicio de dónde ocurrió.

La auditoría incluye además dos medidas de completitud de atributos: cuántos registros declaran conectores y cuántos declaran fecha de verificación. Estas cifras anticipan el diagnóstico del Bloque B, donde la completitud de atributos (y no la precisión posicional) resultó ser el problema dominante de las fuentes abiertas colombianas.

## 11.7. Comportamiento ante ausencia total de registros

Si tras recorrer todos los elementos no se construyó ningún registro, la función interrumpe la ejecución mediante RuntimeError en lugar de devolver un GeoDataFrame vacío.

La decisión es coherente con el tratamiento dado a la lista vacía en la etapa de descarga: un inventario sin registros no admite continuación razonable, y devolverlo permitiría que el vacío se propagara sin ser advertido hasta manifestarse mucho más adelante.

In [14]:
def transformar_elementos_osm(
    elementos,
    ciudad,
    fecha_descarga,
    timestamp_osm_base=None,
    fecha_corte_osm=None,
):
    """
    Convierte los elementos recuperados desde Overpass
    en un GeoDataFrame.

    Correcciones respecto de la versión anterior:

    1. Se conserva el diccionario 'tags' completo en tags_json.
       La versión anterior seleccionaba nueve etiquetas y
       descartaba el resto, lo que obligaría a redescargar
       al necesitar cualquier otra.

    2. Se aísla el subconjunto socket:* en sockets_json.
       Es el equivalente en OSM del tipo de conector y la
       potencia por conector de OpenChargeMap, y sin él la
       reconciliación de atributos del Bloque B no es posible.

    3. Se incorpora check_date, análogo directo de
       fecha_ultima_verificacion de OpenChargeMap.

    4. Se incorporan los campos de meta (version, timestamp,
       changeset, user).

    Los campos JSON se guardan como cadena, no como dict,
    porque el driver GeoJSON no admite objetos anidados.
    """

    registros = []

    registros_sin_coordenadas = 0

    elementos_tipo_no_valido = 0

    for elemento in elementos:

        tipo = elemento.get(
            "type"
        )

        osm_id = elemento.get(
            "id"
        )

        if tipo not in {
            "node",
            "way",
            "relation",
        }:

            elementos_tipo_no_valido += 1

            continue

        tags = elemento.get(
            "tags",
            {}
        )

        latitud = elemento.get(
            "lat"
        )

        longitud = elemento.get(
            "lon"
        )

        # ---------------------------------------------------------
        # Way y relation entregan coordenadas mediante center
        # ---------------------------------------------------------

        if (
            latitud is None
            or longitud is None
        ):

            center = elemento.get(
                "center",
                {}
            )

            latitud = center.get(
                "lat"
            )

            longitud = center.get(
                "lon"
            )

        if (
            latitud is None
            or longitud is None
        ):

            registros_sin_coordenadas += 1

            continue

        # ---------------------------------------------------------
        # Conectores: todas las claves socket:*
        # ---------------------------------------------------------

        sockets = {
            clave: valor
            for clave, valor in tags.items()
            if clave.startswith("socket:")
        }

        registros.append(
            {
                "osm_uid":
                    f"{tipo}/{osm_id}",

                "ciudad":
                    ciudad,

                "osm_tipo":
                    tipo,

                "osm_id":
                    osm_id,

                # --- identificación ---

                "nombre":
                    tags.get("name"),

                "operador":
                    tags.get("operator"),

                "marca":
                    tags.get("brand"),

                "red":
                    tags.get("network"),

                # --- dirección ---

                "direccion":
                    tags.get("addr:street"),

                "numero":
                    tags.get(
                        "addr:housenumber"
                    ),

                # --- atributos de ruteo ---

                "capacidad":
                    tags.get("capacity"),

                "potencia":
                    tags.get(
                        "charging_station:output"
                    ),

                "acceso":
                    tags.get("access"),

                "cobro":
                    tags.get("fee"),

                "horario":
                    tags.get(
                        "opening_hours"
                    ),

                "n_sockets_declarados":
                    len(sockets),

                "sockets_json":
                    json.dumps(
                        sockets,
                        ensure_ascii=False,
                    ),

                # --- vigencia ---

                "check_date":
                    tags.get("check_date"),

                "survey_date":
                    tags.get("survey:date"),

                # --- meta de OSM ---

                "osm_version":
                    elemento.get("version"),

                "osm_timestamp":
                    elemento.get("timestamp"),

                "osm_changeset":
                    elemento.get("changeset"),

                "osm_user":
                    elemento.get("user"),

                # --- respaldo íntegro ---

                "n_tags":
                    len(tags),

                "tags_json":
                    json.dumps(
                        tags,
                        ensure_ascii=False,
                        sort_keys=True,
                    ),

                # --- coordenadas y procedencia ---

                "latitud":
                    float(latitud),

                "longitud":
                    float(longitud),

                "fuente":
                    "OpenStreetMap",

                "licencia":
                    "ODbL",

                "fecha_descarga":
                    fecha_descarga,
                    
                # Corte del mapa al que corresponde el registro.
                # fecha_descarga es el reloj local; esto es el
                # estado del mundo que el dato describe, y viaja
                # dentro del GeoJSON para que no dependa de que
                # alguien abra el archivo de metadatos.

                "timestamp_osm_base":
                    timestamp_osm_base,

                "fecha_corte_osm":
                    fecha_corte_osm,
            }
        )

    if not registros:

        raise RuntimeError(
            f"No se pudieron construir "
            f"registros para {ciudad}."
        )

    df = pd.DataFrame(
        registros
    )

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(
            df["longitud"],
            df["latitud"],
        ),
        crs=CRS_WGS84,
    )

    auditoria = {
        "elementos_recibidos":
            len(elementos),

        "registros_transformados":
            len(registros),

        "registros_sin_coordenadas":
            registros_sin_coordenadas,

        "elementos_tipo_no_valido":
            elementos_tipo_no_valido,

        "registros_con_sockets":
            int(
                (df["n_sockets_declarados"] > 0)
                .sum()
            ),

        "registros_con_check_date":
            int(
                df["check_date"]
                .notna()
                .sum()
            ),

        "registros_con_survey_date":
            int(
                df["survey_date"]
                .notna()
                .sum()
            ),
    }

    return gdf, auditoria

# 12. Validación de coordenadas

Construido el GeoDataFrame, validar_coordenadas comprueba que las coordenadas sean geográficamente admisibles y que concuerden con la geometría efectivamente almacenada.

La función opera sobre dos frentes independientes: el dominio de los valores y la correspondencia entre las columnas numéricas y el objeto geométrico. El segundo es el que atiende el modo de fallo específico que motivó el rediseño del Bloque A.

## 12.1. Comprobación de dominio

Se verifica que todas las latitudes se sitúen entre −90 y 90 grados, y todas las longitudes entre −180 y 180. Cualquier valor fuera de esos rangos interrumpe la ejecución mediante ValueError con el nombre de la ciudad afectada.

La comprobación se realiza sobre la columna completa mediante operaciones vectorizadas, y se exige que la condición se cumpla en todos los registros. No se admite un fallo parcial: una sola coordenada inválida detiene el proceso.

Como se señaló al documentar validar_bbox, esta clase de verificación captura la inversión de latitud y longitud únicamente cuando produce valores fuera de dominio. Una latitud confundida con longitud en Colombia toma valores cercanos a −74, que exceden el rango de latitud y quedan detectados aquí; el caso inverso, en cambio, produce valores en torno a 4 o 6 grados, admisibles como longitud, y no sería capturado por rango. La sección siguiente cubre parte de ese vacío por otra vía.

## 12.2. Correspondencia entre columnas y geometría

La segunda comprobación es la que da sentido principal a esta función. Se calcula la diferencia absoluta entre las coordenadas del objeto geométrico —geometry.x y geometry.y— y las columnas numéricas longitud y latitud, y se exige que ambas diferencias no superen una tolerancia de 1e-10.

El GeoDataFrame almacena la posición dos veces: como par de columnas numéricas y como objeto geométrico. La geometría se construyó a partir de esas columnas, de modo que en principio deben coincidir. Verificarlo comprueba que efectivamente lo hagan.

Por qué esta verificación existe. El error crítico del Bloque A no fue de cálculo sino de correspondencia: la fila de OpenStreetMap que se unía a cada registro no era la que había producido la distancia mínima. Un desalineamiento de índices hacía que el dato reportado y el dato usado en el cálculo pertenecieran a registros distintos. El fallo era invisible porque las cifras resultantes eran plausibles.

Esta comprobación es la salvaguarda análoga en el punto donde el problema puede originarse. Un desalineamiento entre las columnas y la geometría —producido, por ejemplo, por una reordenación o un filtrado que afecte a una y no a la otra— desplazaría los puntos sin generar ningún error visible: el archivo se exportaría correctamente, los mapas mostrarían estaciones, y solo las distancias resultarían equivocadas.

Por qué se verifica y no se asume. Podría objetarse que la geometría se construye inmediatamente antes, en la misma función, y que por tanto la coincidencia está garantizada. La objeción es correcta hoy, pero el criterio de trabajo del proyecto exige que todo resultado importante se compruebe por un camino distinto al que lo produjo. Una verificación cuyo valor depende de que nadie modifique el código intermedio es precisamente la que conviene dejar escrita.

## 12.3. Sobre la tolerancia

La comparación no exige igualdad exacta sino una diferencia inferior a 1e-10 grados.

El margen atiende la representación de punto flotante: la conversión de un valor a geometría y su lectura posterior pueden introducir diferencias en el último bit significativo, sin que haya error alguno. Exigir igualdad estricta produciría fallos espurios.

La magnitud del margen es despreciable en términos geográficos: 1e-10 grados corresponde a una fracción de milímetro sobre la superficie terrestre. Es, por tanto, lo bastante amplio para tolerar el ruido numérico y lo bastante estrecho para detectar cualquier desplazamiento con significado espacial.

## 12.4. Alcance

La función verifica que las coordenadas sean admisibles y consistentes con la geometría. No verifica que sean correctas.

Una estación cuya coordenada esté mal registrada en OpenStreetMap —situada en otra ciudad, o desplazada varias manzanas— pasaría todas estas comprobaciones sin objeción, porque el valor es admisible y la geometría lo refleja fielmente. Esa clase de error corresponde al filtro espacial contra el límite municipal y, más adelante, al contraste con la segunda fuente.

Lo que esta validación garantiza es que el dato interno del GeoDataFrame es coherente consigo mismo, condición previa a cualquier afirmación sobre su exactitud.

In [15]:
def validar_coordenadas(
    estaciones,
    ciudad,
):
    """
    Valida las coordenadas geográficas y su correspondencia
    con la geometría almacenada.
    """

    latitudes_validas = (
        estaciones["latitud"]
        .between(-90, 90)
    )

    longitudes_validas = (
        estaciones["longitud"]
        .between(-180, 180)
    )

    if not latitudes_validas.all():

        raise ValueError(
            f"{ciudad}: existen latitudes inválidas."
        )

    if not longitudes_validas.all():

        raise ValueError(
            f"{ciudad}: existen longitudes inválidas."
        )

    diferencia_longitud = (
        estaciones.geometry.x
        - estaciones["longitud"]
    ).abs()

    diferencia_latitud = (
        estaciones.geometry.y
        - estaciones["latitud"]
    ).abs()

    tolerancia = 1e-10

    if (
        diferencia_longitud
        > tolerancia
    ).any():

        raise ValueError(
            f"{ciudad}: existe inconsistencia "
            "entre longitud y geometría."
        )

    if (
        diferencia_latitud
        > tolerancia
    ).any():

        raise ValueError(
            f"{ciudad}: existe inconsistencia "
            "entre latitud y geometría."
        )

    return True

# 13. Delimitación mediante límites oficiales

Validado el inventario en su consistencia interna, se procede a recortarlo espacialmente. La función cargar_limite_ciudad prepara el insumo de ese recorte: lee el archivo de límites, selecciona el municipio correspondiente, lo transforma al sistema de referencia de trabajo y lo reduce a una geometría única.

La función devuelve dos objetos: el límite ya procesado y una ficha con su procedencia. Esa segunda salida responde a una particularidad del insumo que se explica a continuación.

## 13.1. Un insumo que no se descarga por programa

El criterio general de trabajo del proyecto exige que las fuentes se descarguen mediante código, con la fecha registrada, de modo que cualquier persona pueda reejecutar el cuaderno desde cero. El archivo de límites municipales no cumple esa condición: se obtiene manualmente desde el portal Colombia en Mapas y se aporta junto con el cuaderno.

El mismo criterio prevé este caso y establece cómo tratarlo: si un paso exige una acción manual, debe quedar documentado como tal. La función implementa esa exigencia convirtiéndola en requisito de ejecución.

La procedencia no es un parámetro opcional ni un comentario: es un argumento obligatorio cuya estructura se valida antes de leer el archivo. Se comprueba que contenga los cuatro campos que identifican el origen del insumo —entidad, URL, fecha de descarga y método de obtención— y se rechaza cualquier campo cuyo valor sea el marcador de pendiente. La función no puede ejecutarse sobre un archivo cuyo origen no esté declarado.

La ficha devuelta traslada esa información a los metadatos del entregable, de modo que la acción manual queda registrada en el producto y no solo en el código que lo generó.

## 13.2. Selección del municipio dentro de la capa

La capa del IGAC es nacional: contiene los 1.105 municipios del país. La selección del municipio de interés se realiza mediante los parámetros columna_codigo y codigo.

Selección por código y no por nombre. El criterio de búsqueda es el código DANE. En esta capa el municipio 11001 figura con la denominación "Bogotá, D.C.", con coma y abreviatura; una búsqueda por texto tendría que anticipar esa forma exacta, además de la tilde y la capitalización, y devolvería un conjunto vacío ante cualquier discrepancia. El código es un identificador estable y no está sujeto a variantes de escritura.

Tres comprobaciones sobre el filtro. Se verifica que la columna de código exista en la capa —y, si no, el mensaje de error enumera las columnas disponibles, para no obligar a inspeccionar el archivo por separado—; que el código encuentre al menos un registro; y que encuentre exactamente uno.

Esta última comprobación merece explicitarse. Si una versión futura de la capa representara un municipio en varias filas, la función lo disolvería sin objeción y el resultado sería correcto por accidente. Exigir un único registro convierte ese cambio de estructura en un fallo visible en lugar de un comportamiento silencioso.

La guarda para capas multirregistro. Cuando no se entregan los parámetros de filtro y la capa contiene más de un registro, la función interrumpe la ejecución.

Esta guarda previene el fallo más grave que el cambio de fuente podía introducir. Sin ella, una capa nacional sin filtrar se disolvería completa y produciría el contorno del país entero como límite de recorte. Ese límite no genera error alguno: se aplica correctamente, deja pasar todas las estaciones descargadas, y el resultado parece razonable. El filtro espacial existiría formalmente y no filtraría nada. Es la clase de defecto que motivó el criterio de verificación independiente adoptado tras el Bloque A, y por eso se bloquea de manera explícita.

Los parámetros se declaran opcionales para preservar la compatibilidad con archivos que contengan un solo municipio, como los empleados en la versión anterior del cuaderno.

## 13.3. Comprobaciones sobre el archivo

Antes de utilizarlo, el límite atraviesa tres verificaciones.

Se comprueba que la capa no esté vacía. Un archivo legible pero sin registros produciría un recorte que descartaría el inventario completo.

Se comprueba que el sistema de referencia esté definido. Un GeoDataFrame sin CRS no puede reproyectarse, y asumir uno implícitamente introduciría un error de coordenadas indetectable. La verificación se realiza antes de reproyectar, y el valor original se conserva en la ficha: documenta que la transformación ocurrió y desde qué sistema, dato relevante dado que la capa del IGAC se distribuye en EPSG:9377.

Se comprueba que las geometrías sean válidas. Un polígono con autointersecciones puede producir resultados erróneos en las operaciones espaciales sin generar excepción.

Conviene señalar que la validez se evalúa después de reproyectar. Es el estado que importa, puesto que en ese sistema se aplicará el recorte, pero implica que una geometría inválida en el sistema original no quedaría registrada como tal.

## 13.4. Disolución a una geometría única

El límite se reduce a un solo polígono mediante union_all.

La razón es operativa y atañe al recorte que sigue. Un municipio puede representarse mediante varias geometrías; si el recorte espacial se ejecutara contra un conjunto de polígonos en lugar de contra uno solo, una estación situada en la frontera entre dos de ellos podría emparejarse con ambos y aparecer duplicada en el resultado.

Esa duplicación no se manifestaría como error. El inventario simplemente tendría más registros de los que corresponden, y el exceso resultaría indistinguible de una diferencia real de cobertura entre fuentes. Disolver previamente elimina la posibilidad.

El resultado se devuelve como un GeoDataFrame de una sola fila, en lugar de como geometría suelta, para que conserve su sistema de referencia y pueda emplearse directamente en operaciones espaciales.

Esa fila lleva una única columna de atributo, `codigo_dane_limite`, con el código del municipio seleccionado. La elección del contenido es deliberada. Al ejecutarse el cruce espacial de la sección 14, las columnas del límite se incorporan a cada estación emparejada, de modo que ese código viaja al inventario y queda registrado fila por fila.

Se emplea el código DANE y no el nombre del municipio por dos razones. El código es un identificador estable, no sujeto a variantes de escritura, mientras que el nombre reproduciría con otra ortografía —tildes y abreviaturas incluidas— la información que la columna `ciudad` ya contiene, creando dos campos que dicen lo mismo de formas distintas. Y el código aporta un dato que el inventario no tenía: contra qué municipio de la capa nacional del IGAC se validó cada estación, que es la constancia de la delimitación aplicada.

## 13.5. La ficha de procedencia

El segundo objeto devuelto reúne la identificación del archivo y la capa, el sistema de referencia original, el número de registros que contenía la capa antes del filtro, la columna y el código empleados para seleccionar, cuántas geometrías quedaron tras filtrar, y los cuatro campos de procedencia aportados.

El conjunto permite reconstruir exactamente qué se cargó y de dónde. El registro del número de registros de la capa junto con el código filtrado documenta que la selección efectivamente ocurrió: un lector puede constatar que se partió de 1.105 municipios y se conservó uno, sin necesidad de abrir el archivo.

## 13.6. Ejecución

La función se invoca una vez por ciudad, con el mismo archivo y la misma capa, y con el código DANE correspondiente tomado del diccionario de configuración. Que ambas ciudades compartan insumo es consecuencia directa de la sustitución documentada en la sección 6: la capa nacional del IGAC reemplaza dos archivos de procedencia distinta (uno distrital, otro derivado de OpenStreetMap) por una fuente única, oficial e independiente de la fuente de estaciones en ambos casos.

In [16]:
def cargar_limite_ciudad(
    archivo,
    capa,
    nombre_ciudad,
    procedencia,
    columna_codigo=None,
    codigo=None,
):
    """
    Carga el límite oficial de la ciudad y lo transforma
    a EPSG:4326.

    El archivo NO se descarga por programa: se aporta junto
    con el cuaderno. Por eso su procedencia se exige de forma
    explícita y se propaga a los metadatos del entregable,
    como acción manual documentada.

    columna_codigo y codigo permiten seleccionar un municipio
    dentro de una capa nacional. Son obligatorios cuando la
    capa trae más de un registro: la del IGAC tiene 1.105
    municipios y sin filtrar se disolverían todos en una sola
    geometría que abarcaría el país entero.

    El límite se disuelve en una sola geometría para que el
    filtrado espacial no genere filas duplicadas.
    """

    campos_procedencia = {
        "entidad",
        "url",
        "fecha_descarga",
        "obtencion",
    }

    faltantes = (
        campos_procedencia
        - set(procedencia.keys())
    )

    if faltantes:

        raise ValueError(
            f"{nombre_ciudad}: la procedencia del límite "
            f"está incompleta: {faltantes}"
        )

    sin_completar = [
        campo
        for campo, valor in procedencia.items()
        if str(valor).strip().upper() == "COMPLETAR"
    ]

    if sin_completar:

        raise ValueError(
            f"{nombre_ciudad}: la procedencia del límite "
            f"tiene campos sin diligenciar: {sin_completar}. "
            "El archivo se aporta manualmente, de modo que "
            "su origen debe quedar documentado."
        )

    limite = gpd.read_file(
        archivo,
        layer=capa,
    )

    if limite.empty:

        raise ValueError(
            f"El límite de {nombre_ciudad} "
            "está vacío."
        )

    # ------------------------------------------------------------------------
    # Selección del municipio dentro de la capa
    # ------------------------------------------------------------------------
    # Se filtra por código y no por nombre: en la capa del IGAC
    # Bogotá figura como "Bogotá, D.C.", con coma y abreviatura,
    # de modo que la búsqueda por texto es frágil.

    registros_originales = len(limite)

    if columna_codigo is not None and codigo is not None:

        if columna_codigo not in limite.columns:

            raise ValueError(
                f"{nombre_ciudad}: la capa '{capa}' no tiene "
                f"la columna '{columna_codigo}'. Columnas "
                f"disponibles: {list(limite.columns)}"
            )

        limite = limite[
            limite[columna_codigo] == codigo
        ]

        if limite.empty:

            raise ValueError(
                f"{nombre_ciudad}: no hay registros con "
                f"{columna_codigo} = {codigo!r} en la capa "
                f"'{capa}'."
            )

        if len(limite) != 1:

            raise ValueError(
                f"{nombre_ciudad}: se esperaba exactamente "
                f"un registro con {columna_codigo} = {codigo!r}, "
                f"se encontraron {len(limite)}."
            )

    elif registros_originales > 1:

        # Sin filtro, una capa multirregistro se disolvería
        # entera en un solo polígono sin emitir error.

        raise ValueError(
            f"{nombre_ciudad}: la capa '{capa}' tiene "
            f"{registros_originales} registros y no se "
            "indicó columna_codigo ni codigo. Disolverla "
            "completa produciría un límite incorrecto."
        )

    if limite.crs is None:

        raise ValueError(
            f"El límite de {nombre_ciudad} "
            "no tiene CRS definido."
        )

    crs_original = limite.crs.to_string()

    limite = limite.to_crs(
        CRS_WGS84
    )

    if not limite.geometry.is_valid.all():

        raise ValueError(
            f"El límite de {nombre_ciudad} "
            "contiene geometrías inválidas."
        )

    # ------------------------------------------------------------------------
    # Disolver a una sola geometría
    # ------------------------------------------------------------------------

    geometria_unica = limite.union_all()

    limite_disuelto = gpd.GeoDataFrame(
        {
            "codigo_dane_limite": [codigo]
        },
        geometry=[
            geometria_unica
        ],
        crs=CRS_WGS84,
    )

    ficha = {
        "ciudad": nombre_ciudad,
        "archivo": archivo,
        "capa": capa,
        "crs_original": crs_original,
        "registros_en_la_capa": registros_originales,
        "columna_codigo": columna_codigo,
        "codigo_filtrado": codigo,
        "geometrias_originales": len(limite),
        "entidad": procedencia["entidad"],
        "url": procedencia["url"],
        "fecha_descarga_limite": (
            procedencia["fecha_descarga"]
        ),
        "obtencion": procedencia["obtencion"],
    }

    return limite_disuelto, ficha

In [17]:
# --- Bogotá ---
limite_bog, ficha_bog = cargar_limite_ciudad(
    archivo=LIMITE_IGAC["archivo"],
    capa=LIMITE_IGAC["capa"],
    nombre_ciudad=AREAS_ESTUDIO["Bogota"]["nombre"],
    procedencia=LIMITE_IGAC["procedencia"],
    columna_codigo=LIMITE_IGAC["columna_codigo"],
    codigo=LIMITE_IGAC["codigos_dane"]["Bogota"],
)

# --- Medellín ---
limite_med, ficha_med = cargar_limite_ciudad(
    archivo=LIMITE_IGAC["archivo"],
    capa=LIMITE_IGAC["capa"],
    nombre_ciudad=AREAS_ESTUDIO["Medellin"]["nombre"],
    procedencia=LIMITE_IGAC["procedencia"],
    columna_codigo=LIMITE_IGAC["columna_codigo"],
    codigo=LIMITE_IGAC["codigos_dane"]["Medellin"],
)

for ficha in (ficha_bog, ficha_med):
    print(
        f"{ficha['ciudad']}: código {ficha['codigo_filtrado']} | "
        f"{ficha['registros_en_la_capa']} registros en la capa | "
        f"CRS origen {ficha['crs_original']}"
    )

Bogotá D.C.: código 11001 | 1105 registros en la capa | CRS origen EPSG:9377
Medellín: código 05001 | 1105 registros en la capa | CRS origen EPSG:9377


# 14. Filtrado por límite oficial

Cargado el límite, filtrar_por_limite_oficial aplica el recorte espacial: conserva únicamente las estaciones situadas dentro del municipio y descarta las que quedaron incluidas en el bbox de consulta pero pertenecen a otro.

Es el paso donde el límite oficial cumple su función de verificación externa. Cada estación conservada ha sido corroborada por una fuente que no participó en su registro.

## 14.1. Reproyección defensiva

La función comienza comprobando que ambos objetos compartan sistema de referencia y, de no ser así, reproyecta el límite al de las estaciones.

La comprobación es redundante en el flujo actual: cargar_limite_ciudad entrega el límite en EPSG:4326 y transformar_elementos_osm construye las estaciones en ese mismo sistema. Su valor está en que no depende de esa coincidencia.

Una operación espacial entre capas de sistemas distintos no siempre genera error. Según la combinación, puede devolver un resultado vacío —que se interpretaría como ausencia de estaciones en el municipio— o un conjunto arbitrario. Verificarlo aquí evita que un cambio en cualquiera de las dos funciones anteriores se propague como un resultado silenciosamente equivocado.

Se reproyecta el límite y no las estaciones, deliberadamente: reproyectar las estaciones desplazaría sus coordenadas y rompería la correspondencia con las columnas latitud y longitud que validar_coordenadas acaba de comprobar.

## 14.2. El predicado espacial

El cruce se realiza mediante sjoin con predicado intersects y unión interna, de modo que solo sobreviven las estaciones que intersectan el polígono municipal.

intersects incluye los puntos situados exactamente sobre la frontera, a diferencia de within, que los excluiría. La elección es coherente con la advertencia que el IGAC declara sobre su propia capa: la entidad señala que el producto no es apropiado para la ubicación exacta de puntos. Ante un dato cuya exactitud en el borde no está garantizada por su productor, el criterio inclusivo es el prudente. Una estación descartada por caer unos metros del lado equivocado de una frontera cuya precisión se declara insuficiente sería una pérdida injustificada.

Conviene declarar el efecto: el recorte es ligeramente permisivo en la frontera. Si alguna estación resultara conservada por esta razón, correspondería identificarla y reportarla como caso indeterminado, según lo señalado en la sección 6.4, y esa identificación la ejecuta identificar_casos_frontera, que mide la distancia de cada estación al borde municipal y reporta las que quedan a menos de cincuenta metros

## 14.3. Limpieza posterior al cruce

Tras el sjoin se ejecutan dos operaciones de saneamiento.

Eliminación de index_right. El cruce añade una columna con el índice del registro del límite con el que cada estación se emparejó. Es información de la operación, no del inventario, y arrastrarla contaminaría el archivo exportado con un campo sin significado para quien lo lea. Se elimina previa comprobación de que exista, ya que su presencia depende de la versión de la biblioteca.
Columnas que aporta el cruce. El `sjoin` incorpora a cada estación las columnas del límite con el que se emparejó, y las dos resultantes tienen destinos distintos.

`index_right` contiene el índice de la fila del límite. Es información de la operación, no del inventario, y arrastrarla contaminaría el archivo exportado con un campo sin significado para quien lo lea. Se elimina con `errors="ignore"`, ya que su presencia depende de la versión de la biblioteca.

`codigo_dane_limite` sí se conserva. A diferencia de la anterior, no es un residuo del mecanismo del cruce sino su resultado sustantivo: deja constancia, en cada registro, de contra qué municipio de la capa nacional del IGAC se validó esa estación. La sección 15 la incorpora a las columnas exigidas del inventario, y la validación del GeoJSON de la sección 18 comprueba sobre el archivo escrito que todos los registros declaren un único código y que sea el que la configuración pedía.

Eliminación de duplicados por osm_uid. Aunque el límite se disolvió previamente en una geometría única, la deduplicación se conserva como salvaguarda. Un sjoin contra varias geometrías produce una fila por cada emparejamiento, de modo que una estación en la frontera entre dos polígonos aparecería dos veces.

La duplicación resultante no se manifestaría como error: el inventario tendría simplemente más registros de los que corresponden, y el exceso sería indistinguible de una diferencia real de cobertura. Como el identificador osm_uid combina tipo e identificador de OpenStreetMap y es único por elemento, sirve de criterio inequívoco para detectar la repetición.

La verificación es doble y deliberada. La disolución en cargar_limite_ciudad previene el problema; esta deduplicación lo captura si la prevención fallara. Es la aplicación del criterio de comprobar por un camino distinto al que produjo el resultado.

Finalmente se reindexa el resultado. El sjoin conserva los índices originales, que tras el filtrado quedan discontinuos; un índice con huecos es precisamente la condición que produjo el desalineamiento del Bloque A, donde la fila unida no era la que había generado el cálculo. Reindexar cierra esa posibilidad antes de que el inventario continúe hacia las etapas siguientes.

In [18]:
def filtrar_por_limite_oficial(estaciones, limite):
    """
    Separa las estaciones que caen dentro del límite oficial de
    las que quedaron dentro del bbox pero pertenecen a otro
    municipio.

    Devuelve las dos partes y no solo la primera. Las excluidas no
    son un residuo del procesamiento: son estaciones correctamente
    registradas en municipios vecinos, y su conteo informa sobre la
    infraestructura de carga del borde metropolitano.
    """

    if estaciones.crs != limite.crs:
        limite = limite.to_crs(estaciones.crs)

    estaciones_filtradas = gpd.sjoin(
        estaciones,
        limite,
        how="inner",
        predicate="intersects",
    )

    estaciones_filtradas = estaciones_filtradas.drop(
        columns="index_right",
        errors="ignore",
    )

    estaciones_filtradas = (
        estaciones_filtradas
        .drop_duplicates(subset=["osm_uid"])
        .reset_index(drop=True)
    )

    excluidas = estaciones[
        ~estaciones["osm_uid"].isin(
            estaciones_filtradas["osm_uid"]
        )
    ].reset_index(drop=True)

    return estaciones_filtradas, excluidas


def identificar_casos_frontera(
    estaciones,
    limite,
    umbral_metros=50.0,
):
    """
    Mide la distancia de cada estación al BORDE del municipio y
    marca las que quedan por debajo del umbral.

    Es la segunda consecuencia operativa de la restricción que el
    IGAC declara sobre su propia capa: al usar 'intersects' el
    recorte es deliberadamente permisivo en la frontera, y las
    estaciones conservadas por esa razón deben identificarse
    individualmente como casos indeterminados.

    Ojo con la diferencia respecto de la prueba 8: aquella mide
    distancia al POLÍGONO, y da cero para todo punto interior.
    Esta mide distancia al BORDE, que es lo que dice qué tan cerca
    está cada estación de la frontera cuya exactitud la entidad
    productora no garantiza.
    """

    estaciones_m = estaciones.to_crs(CRS_METRICO)

    borde = (
        limite.to_crs(CRS_METRICO)
        .geometry.iloc[0]
        .boundary
    )

    distancias = estaciones_m.geometry.distance(borde)

    resultado = estaciones.copy()

    resultado["distancia_al_borde_m"] = distancias.values.round(2)

    frontera = (
        resultado[
            resultado["distancia_al_borde_m"] < umbral_metros
        ]
        .sort_values("distancia_al_borde_m")
        .reset_index(drop=True)
    )

    return resultado, frontera


def clasificar_excluidas_por_municipio(
    excluidas,
    archivo,
    capa,
    columna_codigo,
    columna_nombre="MpNombre",
):
    """
    Determina a qué municipio pertenece cada estación descartada
    por el filtro, cruzándola contra la misma capa nacional del
    IGAC que produjo el recorte.

    Convierte un descarte invisible en una observación sobre la
    distribución territorial de la infraestructura de carga.
    """

    if excluidas.empty:
        return excluidas

    capa_nacional = gpd.read_file(
        archivo,
        layer=capa,
    ).to_crs(CRS_WGS84)

    columnas = [columna_codigo, columna_nombre, "geometry"]

    faltantes = [
        c for c in columnas if c not in capa_nacional.columns
    ]

    if faltantes:
        raise ValueError(
            f"La capa '{capa}' no tiene las columnas {faltantes}."
        )

    unidas = gpd.sjoin(
        excluidas,
        capa_nacional[columnas],
        how="left",
        predicate="intersects",
    )

    unidas = (
        unidas
        .drop(columns="index_right", errors="ignore")
        .drop_duplicates(subset=["osm_uid"])
        .reset_index(drop=True)
        .rename(
            columns={
                columna_codigo: "municipio_codigo",
                columna_nombre: "municipio_nombre",
            }
        )
    )

    return unidas

# 15. Validación estructural del inventario

Aplicado el recorte espacial, validar_inventario_osm somete el inventario a un conjunto de comprobaciones estructurales antes de exportarlo. Es la última verificación sobre el dato en memoria.

La función agrupa validaciones sobre presencia, unicidad, tipos, geometría, sistema de referencia y procedencia. Todas interrumpen la ejecución mediante ValueError con el nombre de la ciudad; ninguna corrige ni descarta. La decisión es coherente con el resto del cuaderno: un inventario que no satisface estas condiciones no debe exportarse, y una corrección automática ocultaría el problema en lugar de resolverlo.

## 15.1. Inventario vacío

La primera comprobación es que el inventario tenga registros.

Su ubicación importa: se ejecuta después del recorte espacial, que es el punto donde el inventario puede quedar vacío sin que ninguna función anterior lo advierta. filtrar_por_limite_oficial devuelve un GeoDataFrame vacío sin objeción si ninguna estación cae dentro del municipio, situación que se produciría si el bbox apuntara al lugar equivocado, si el código DANE seleccionara otro municipio, o si el cruce espacial se ejecutara entre sistemas de referencia incompatibles.

Ninguno de esos fallos genera excepción por sí mismo. Todos desembocan en el mismo resultado: cero estaciones. Esta comprobación los convierte a todos en un error visible.

## 15.2. Presencia de columnas

Se verifica que existan las nueve columnas mínimas que las etapas posteriores requieren: identificadores, coordenadas, geometría, fuente, fecha de descarga y el código DANE del municipio contra el que se validó cada registro.

La inclusión de `codigo_dane_limite` entre las columnas exigidas es consecuencia de la sustitución documentada en la sección 6. Al provenir del cruce espacial, esa columna podría desaparecer del inventario si el límite dejara de aportarla —por ejemplo, si una modificación futura de `cargar_limite_ciudad` construyera el GeoDataFrame del límite sin ella— sin que ninguna otra comprobación lo advirtiera. Exigirla aquí convierte esa pérdida en un fallo visible.

La comprobación se realiza por diferencia de conjuntos, de modo que se reportan todas las ausencias en un solo mensaje. Las columnas faltantes producirían fallos en la exportación o en los bloques siguientes, lejos del punto donde se originaron.

## 15.3. Identificadores

Se comprueba que ningún registro carezca de osm_id y que no existan osm_uid repetidos.

Sobre la unicidad. El identificador osm_uid combina el tipo y el número de elemento, y es único por construcción dentro de OpenStreetMap. Que no lo sea aquí solo puede deberse a un problema del procesamiento: una duplicación introducida por el cruce espacial, o una respuesta anómala del servidor.

La comprobación es redundante respecto de la deduplicación aplicada en el filtrado, y esa redundancia es deliberada. Una verificación cuya utilidad depende de que la función anterior siga comportándose como hoy es precisamente la que conviene mantener escrita.

Conviene distinguir este control de otro que el proyecto tiene pendiente. Aquí se verifica que un mismo elemento de OpenStreetMap no aparezca dos veces en el inventario. La existencia de duplicados internos en cada fuente —dos elementos distintos que describen la misma estación física— es un problema diferente, que corresponde al Bloque B y que esta validación no aborda.

## 15.4. Tipos de elemento

Se verifica que los valores de osm_tipo pertenezcan al conjunto node, way y relation.

La comprobación se realiza también por diferencia de conjuntos, lo que permite enumerar todos los valores inesperados en lugar de detenerse en el primero. Es la contraparte, sobre el inventario ya construido, de la comprobación equivalente que transformar_elementos_osm aplica sobre cada elemento recibido.

## 15.5. Coordenadas y geometría

La validación de coordenadas se delega en validar_coordenadas, documentada en la sección 12, que comprueba el dominio de los valores y su correspondencia con la geometría almacenada.

La reutilización tiene un efecto relevante: esa correspondencia se verifica dos veces, una sobre el inventario recién transformado y otra sobre el inventario ya filtrado. Es exactamente en el intervalo entre ambas donde ocurren las operaciones capaces de romperla —el cruce espacial, la deduplicación, la reindexación— de modo que la segunda ejecución no es una repetición ociosa sino la comprobación de que esas operaciones no desalinearon los datos.

Sobre la geometría se añaden tres verificaciones. Que ninguna sea nula, caso que produciría fallos en operaciones espaciales posteriores. Que ninguna esté vacía, situación distinta de la nulidad y que las operaciones espaciales tratan de manera silenciosa. Y que todas sean de tipo Point, lo que confirma que la homogeneización descrita en la sección 11.1 se aplicó a los tres tipos de elemento y que ninguna geometría extensa sobrevivió sin reducir.

## 15.6. Sistema de referencia

Se comprueba que el CRS esté definido y que sea exactamente el declarado en las constantes.

Es la verificación que faltaba en validar_coordenadas y que allí se señaló como limitación. Su ausencia permitiría que una reproyección accidental pasara desapercibida: las columnas latitud y longitud conservarían sus valores en grados mientras la geometría quedaría en metros, y aunque la comprobación de correspondencia detectaría la discrepancia, el mensaje de error apuntaría al síntoma y no a la causa.

## 15.7. Procedencia

Se verifica que todos los registros declaren OpenStreetMap como fuente.

En un cuaderno dedicado exclusivamente a esa fuente la comprobación parece trivial, pero atiende una exigencia central del proyecto. El Bloque A se abrió por la necesidad de retirar una fuente cuya licencia impedía redistribuir el inventario, y el criterio establecido fue que ningún dato de esa procedencia participe en cálculo, figura o conclusión alguna.

Esta comprobación convierte ese criterio en una condición verificada por el código. Si un registro de otra procedencia se incorporara al inventario, por mezcla accidental de archivos o por un error de concatenación, la exportación se detendría.

## 15.8. El resumen devuelto

Cuando todas las comprobaciones pasan, la función devuelve un diccionario con el balance del inventario.

Además del conteo de registros, incluye dos medidas de unicidad calculadas por caminos distintos: el número de combinaciones únicas de tipo e identificador, y el número de valores únicos de osm_uid. Ambas deben coincidir con el total de registros, y su cálculo independiente permite constatarlo sin recurrir a la misma operación que produjo el resultado.

In [19]:
def validar_inventario_osm(
    estaciones,
    ciudad,
):
    """
    Ejecuta las validaciones estructurales del inventario OSM.
    """

    if estaciones.empty:

        raise ValueError(
            f"El inventario de {ciudad} "
            "está vacío."
        )

    columnas_requeridas = {
        "osm_uid",
        "osm_tipo",
        "osm_id",
        "latitud",
        "longitud",
        "geometry",
        "fuente",
        "fecha_descarga",
        "codigo_dane_limite",
        "timestamp_osm_base",
        "distancia_al_borde_m",
    }

    columnas_faltantes = (
        columnas_requeridas
        - set(estaciones.columns)
    )

    if columnas_faltantes:

        raise ValueError(
            f"{ciudad}: faltan columnas: "
            f"{columnas_faltantes}"
        )

    # ------------------------------------------------------------------------
    # Identificadores
    # ------------------------------------------------------------------------

    if estaciones["osm_id"].isna().any():

        raise ValueError(
            f"{ciudad}: existen registros "
            "sin osm_id."
        )

    if estaciones["osm_uid"].duplicated().any():

        raise ValueError(
            f"{ciudad}: existen osm_uid duplicados."
        )

    # ------------------------------------------------------------------------
    # Tipos OSM
    # ------------------------------------------------------------------------

    tipos_validos = {
        "node",
        "way",
        "relation",
    }

    tipos_encontrados = set(
        estaciones["osm_tipo"].dropna()
    )

    tipos_invalidos = (
        tipos_encontrados
        - tipos_validos
    )

    if tipos_invalidos:

        raise ValueError(
            f"{ciudad}: tipos OSM inválidos: "
            f"{tipos_invalidos}"
        )

    # ------------------------------------------------------------------------
    # Coordenadas
    # ------------------------------------------------------------------------

    validar_coordenadas(
        estaciones,
        ciudad,
    )

    # ------------------------------------------------------------------------
    # Geometrías
    # ------------------------------------------------------------------------

    if estaciones.geometry.isna().any():

        raise ValueError(
            f"{ciudad}: existen geometrías nulas."
        )

    if estaciones.geometry.is_empty.any():

        raise ValueError(
            f"{ciudad}: existen geometrías vacías."
        )

    if estaciones.geometry.geom_type.ne(
        "Point"
    ).any():

        raise ValueError(
            f"{ciudad}: existen geometrías "
            "diferentes de Point."
        )

    # ------------------------------------------------------------------------
    # CRS
    # ------------------------------------------------------------------------

    if estaciones.crs is None:

        raise ValueError(
            f"{ciudad}: CRS no definido."
        )

    if estaciones.crs.to_string() != CRS_WGS84:

        raise ValueError(
            f"{ciudad}: CRS inesperado: "
            f"{estaciones.crs}"
        )

    # ------------------------------------------------------------------------
    # Fuente
    # ------------------------------------------------------------------------

    if (
        estaciones["fuente"]
        .ne("OpenStreetMap")
        .any()
    ):

        raise ValueError(
            f"{ciudad}: existen registros "
            "con una fuente diferente."
        )

    return {
        "ciudad":
            ciudad,

        "registros":
            len(estaciones),

        "elementos_unicos":
            estaciones[
                [
                    "osm_tipo",
                    "osm_id"
                ]
            ]
            .drop_duplicates()
            .shape[0],

        "osm_uid_unicos":
            estaciones[
                "osm_uid"
            ]
            .nunique(),

        "crs":
            estaciones.crs.to_string(),

        "coordenadas_validas":
            True,

        "geometrias_validas":
            True,

        "fuente_valida":
            True,
    }

# 16. Auditoría del recorte espacial

construir_auditoria consolida el balance del proceso completo: cuántos elementos entregó Overpass, cuántos sobrevivieron a cada etapa y cuántos se perdieron en el camino, con el motivo de cada pérdida.

Responde la carencia señalada al documentar el filtrado espacial: la función que aplica el recorte devuelve el inventario resultante pero no informa qué descartó, y esa cifra es un resultado del trabajo y no un residuo del procesamiento.

## 16.1. La cadena de conteos

La auditoría reconstruye el trayecto del inventario mediante cinco cifras encadenadas.

elementos_overpass es lo que el servidor entregó. De ahí se restan los elementos de tipo no reconocido y los que carecían de coordenadas por ambas vías, ambos descartados durante la transformación, para obtener registros_con_coordenadas: el inventario tal como quedó antes del recorte espacial.

Sobre esa base opera el filtrado por límite municipal, del que resulta registros_finales. La diferencia entre ambos es registros_excluidos_por_limite.

La cadena permite leer el trayecto completo del inventario en una sola fila de la tabla de resumen.

Conviene precisar qué comprueba y qué no. `registros_excluidos_por_limite` se calcula restando los registros finales de los que tenían coordenadas, de modo que la suma de las cuatro cifras iguala los elementos recibidos por construcción aritmética y no por verificación. No es un control que pueda fallar: es la descomposición de una resta.

El balance que sí es comprobable ocurre una etapa antes, en la transformación, donde los elementos recibidos, los transformados, los descartados por falta de coordenadas y los descartados por tipo se cuentan de manera independiente dentro del mismo recorrido. Ese contraste se ejecuta como prueba de aceptación número 10: si la suma de los tres destinos no iguala el número de elementos recibidos, hay elementos que el código no está contabilizando y la ejecución se detiene.

## 16.2. Por qué separar las pérdidas por motivo

Los tres motivos de descarte responden a fenómenos distintos y llevan a lecturas opuestas.

Un elemento sin coordenadas es un defecto del registro en OpenStreetMap: existe la estación pero no se puede ubicar. Es un problema de calidad de la fuente.

Un elemento de tipo no reconocido sería un defecto del código o un cambio en la respuesta del servicio. No debería ocurrir, dado que la consulta solicita únicamente los tres tipos válidos.

Un registro excluido por límite no es un defecto de nada. Es una estación correctamente registrada que pertenece a otro municipio y que el bbox alcanzó por su forma rectangular. Es un resultado con valor propio.

Agregarlos en una sola cifra de pérdidas impediría distinguir un problema de calidad de la fuente de un artefacto de la geometría del área consultada.

## 16.3. Los registros excluidos por límite como resultado

Esta cifra merece atención particular en el caso de Bogotá.

Su bbox cubre 338,2 km² situados fuera del Distrito, correspondientes a once municipios vecinos —Soacha, Funza, Cota, Tenjo, Mosquera y otros. Toda estación registrada en esa franja aparece en la respuesta de Overpass y es descartada por el filtro municipal.

El número de estaciones así excluidas informa sobre la infraestructura de carga en el borde metropolitano de Bogotá, y su magnitud relativa indica cuánta cobertura ajena introduce la forma rectangular del área de consulta. En Medellín la situación es distinta: el bbox alcanza los municipios del Área Metropolitana (Bello, Envigado, Itagüí, Sabaneta) y el volumen de exclusiones puede ser considerable.

Reportar la cifra convierte lo que sería un descarte invisible en una observación sobre la distribución territorial de la infraestructura.

16.4. Los porcentajes y su denominador

Se calculan dos proporciones: la fracción del inventario retenida tras el recorte y la excluida por él.

El denominador es registros_con_coordenadas y no elementos_overpass. La elección es deliberada: los porcentajes miden el efecto del recorte espacial, no el del proceso completo. Emplear los elementos recibidos mezclaría en una sola cifra las pérdidas por defecto de la fuente con las exclusiones por pertenencia a otro municipio, que son fenómenos independientes.

La elección del denominador debe declararse al reportar estas cifras. Es la aplicación de un criterio que el proyecto ya adoptó tras la revisión del Bloque A, donde los porcentajes del semáforo se habían calculado sobre 26 registros en lugar de sobre el inventario real, y por tanto no describían lo que aparentaban describir.

Protección contra denominador nulo. Si el inventario previo al recorte estuviera vacío, los porcentajes se fijan en cero en lugar de provocar una división por cero. La situación no debería presentarse, puesto que transformar_elementos_osm interrumpe la ejecución si no logra construir ningún registro, pero la guarda evita que la auditoría falle por un caso que ella misma debería poder reportar.

## 16.5. Conteos absolutos junto a las proporciones

La auditoría entrega tanto las cifras absolutas como los porcentajes.

Es una de las cinco reglas de trabajo del equipo: se reportan conteos absolutos y no solo porcentajes, porque una proporción no informa si se desconoce sobre cuántos casos se calculó. En un inventario del orden de las decenas de estaciones, la diferencia es determinante: un mismo porcentaje puede corresponder a una estación o a varias, y la interpretación cambia por completo.

In [20]:
def construir_auditoria(
    ciudad,
    elementos_overpass,
    registros_con_coordenadas,
    registros_sin_coordenadas,
    registros_finales,
    elementos_tipo_no_valido,
    casos_frontera=0,
    registros_con_survey_date=0,
):
    """
    Construye la auditoría del proceso de extracción
    y delimitación espacial.
    """

    registros_excluidos = (
        registros_con_coordenadas
        - registros_finales
    )

    if registros_con_coordenadas > 0:

        porcentaje_retenido = (
            registros_finales
            / registros_con_coordenadas
        ) * 100

        porcentaje_excluido = (
            registros_excluidos
            / registros_con_coordenadas
        ) * 100

    else:

        porcentaje_retenido = 0
        porcentaje_excluido = 0

    return {

        "ciudad":
            ciudad,

        "elementos_overpass":
            elementos_overpass,

        "registros_con_coordenadas":
            registros_con_coordenadas,

        "registros_sin_coordenadas":
            registros_sin_coordenadas,

        "elementos_tipo_no_valido":
            elementos_tipo_no_valido,

        "registros_finales":
            registros_finales,

        "registros_excluidos_por_limite":
            registros_excluidos,

        "porcentaje_retenido":
            round(
                porcentaje_retenido,
                2,
            ),

        "porcentaje_excluido":
            round(
                porcentaje_excluido,
                2,
            ),

        "casos_frontera":
            casos_frontera,

        "registros_con_survey_date":
            registros_con_survey_date,
    }

# 17. Exportación del entregable

exportar_entregable_a2 produce los tres archivos que constituyen el entregable: el inventario en formato GeoJSON, el texto literal de la consulta ejecutada y un archivo de metadatos.

La estructura responde a una exigencia del proyecto: la descarga debe ser citable y repetible por una persona distinta. Un archivo de datos sin la consulta que lo generó no es reproducible; una consulta sin la constancia de contra qué estado del mapa se ejecutó tampoco lo es. Los tres archivos se producen juntos porque solo juntos sostienen esa afirmación.

## 17.1. El inventario y la consulta

El GeoJSON conserva el inventario ya filtrado y validado, con todas las columnas producidas por la transformación, incluidos los campos tags_json y sockets_json serializados como cadena según lo descrito en la sección 11.5.

La consulta se escribe como texto plano, literal y sin modificaciones. Es el mismo texto cuya huella SHA-256 aparece en los metadatos, de modo que cualquiera puede recalcularla sobre el archivo y constatar que corresponde. La consulta se guarda entera, y no solo su huella, para que sea directamente ejecutable por un tercero sin necesidad de reconstruirla desde el código.

## 17.2. El archivo de metadatos

El archivo de metadatos organiza la información en bloques temáticos.

Licencia y atribución. Se incorpora el texto de atribución exigido por la ODbL, según se declaró en la sección 5.7. La cláusula obliga a acompañar los datos derivados con el reconocimiento de la fuente, y este es el lugar donde ese reconocimiento se materializa.

Reproducibilidad. Reúne la huella de la consulta, el corte temporal solicitado y los cortes reales declarados por el servidor.

La distinción entre lo solicitado y lo obtenido es esencial. fecha_corte_solicitada registra qué se pidió —en la ejecución actual, ningún corte, es decir, el estado vigente del mapa. timestamp_osm_base registra hasta qué momento estaban actualizados los datos del servidor cuando respondió. Solo el segundo permite responder contra qué estado del mapa se hizo la consulta.

El bloque incluye además la constancia del congelamiento: la ruta del archivo crudo, su huella SHA-256 y la indicación de que toda ejecución posterior lo lee de disco. La información viaja en el entregable y no solo en el código, de modo que un lector externo puede verificar contra qué estado del mapa se construyó el inventario sin necesidad de abrir el cuaderno.

Descarga. Consigna el servidor que efectivamente respondió, el identificador del servicio, las marcas temporales locales de inicio y fin, la duración y la zona horaria. Registrar el servidor importa porque los espejos difieren en su desfase de replicación y en si prestan el servicio de datos históricos.

Delimitación. Reproduce el bbox de consulta y el criterio espacial declarado en la configuración. Este último es el que explica, para cada ciudad, cuál de las dos geometrías constituye el recorte vinculante y por qué.

Límite oficial. Documenta el insumo que no se descarga por programa. Se identifica explícitamente como insumo manual en el encabezado del bloque, en cumplimiento del criterio que exige documentar como tal cualquier paso que requiera acción manual.

Auditoría. Traslada el balance completo del proceso: elementos recibidos, descartes por cada motivo, registros finales y las dos proporciones, acompañadas siempre de sus conteos absolutos.

## 17.3. Nomenclatura y valor de los archivos

Los tres archivos incorporan el nombre de la ciudad en su denominación, de modo que las dos áreas de estudio produzcan conjuntos independientes que no se sobrescriban.

El listado final de archivos dentro del propio documento de metadatos permite verificar que el entregable está completo sin recorrer el directorio.

La función devuelve las rutas de los tres archivos, lo que permite a la celda de ejecución consolidarlas y a las pruebas de aceptación releerlas para verificarlas por una vía independiente de la que las produjo.

In [21]:
def exportar_entregable_a2(
    estaciones,
    query,
    resultado_descarga,
    ciudad,
    auditoria,
    bbox,
    criterio_espacial,
    ficha_limite,
    excluidas=None,
    casos_frontera=None,
):
    """
    Exporta los productos correspondientes al entregable A2.
    """

    servidor = (
        resultado_descarga["servidor"]
    )

    archivo_geojson = (
        f"A2_estaciones_OSM_{ciudad}.geojson"
    )

    archivo_query = (
        f"A2_consulta_overpass_{ciudad}.txt"
    )

    archivo_metadata = (
        f"A2_metadata_descarga_{ciudad}.txt"
    )

    archivo_excluidas = (
        f"A2_excluidas_por_limite_{ciudad}.geojson"
    )

    archivo_frontera = (
        f"A2_casos_frontera_{ciudad}.csv"
    )

    # ------------------------------------------------------------------------
    # GeoJSON
    # ------------------------------------------------------------------------

    estaciones.to_file(
        archivo_geojson,
        driver="GeoJSON",
    )

    # ------------------------------------------------------------------------
    # Consulta
    # ------------------------------------------------------------------------

    with open(
        archivo_query,
        "w",
        encoding="utf-8",
    ) as archivo:

        archivo.write(query)

    # ------------------------------------------------------------------------
    # Estaciones descartadas por el límite municipal
    # ------------------------------------------------------------------------
    # No son un residuo: son estaciones de municipios vecinos que el
    # bbox alcanzó por su forma rectangular. Se exportan con el
    # municipio al que pertenecen.

    n_excluidas = 0

    if excluidas is not None and not excluidas.empty:

        excluidas.to_file(
            archivo_excluidas,
            driver="GeoJSON",
        )

        n_excluidas = len(excluidas)

    # ------------------------------------------------------------------------
    # Casos indeterminados en la frontera
    # ------------------------------------------------------------------------

    n_frontera = 0

    if casos_frontera is not None and not casos_frontera.empty:

        (
            casos_frontera
            .drop(columns="geometry", errors="ignore")
            [["osm_uid", "nombre", "distancia_al_borde_m"]]
            .to_csv(archivo_frontera, index=False, encoding="utf-8")
        )

        n_frontera = len(casos_frontera)

    # ------------------------------------------------------------------------
    # Metadata
    # ------------------------------------------------------------------------

    metadata = f"""
ENTREGABLE A2 — METADATOS DE DESCARGA
======================================

Ciudad:
{ciudad}

Fuente:
OpenStreetMap

API:
Overpass API

Etiqueta:
{ETIQUETA_OSM}

LICENCIA Y ATRIBUCIÓN
---------------------
{ATRIBUCION_ODBL}

REPRODUCIBILIDAD Y CONGELAMIENTO
--------------------------------
Huella SHA-256 de la consulta:
{hash_consulta(query)}

Corte temporal solicitado (UTC):
{resultado_descarga["fecha_corte_solicitada"] or "ninguno (estado actual del mapa)"}

Corte temporal en hora local (America/Bogota):
{FECHA_CORTE_LOCAL}

Actualización de la base de datos del servidor (timestamp_osm_base):
{resultado_descarga["timestamp_osm_base"]}
(No es el corte de la consulta: declara hasta qué momento el
servidor tenía sincronizado el mapa mundial cuando se le consultó.
El corte aplicado es el declarado arriba.)

Corte de las áreas (timestamp_areas_base):
{resultado_descarga["timestamp_areas_base"]}

Origen de los datos de esta ejecución:
{resultado_descarga.get("origen", "no declarado")}

Archivo crudo congelado:
{resultado_descarga.get("ruta_crudo", "no producido")}

SHA-256 del archivo crudo:
{resultado_descarga.get("hash_crudo", "no disponible")}

NOTA: el archivo crudo contiene los bytes exactos devueltos por
Overpass. Toda ejecución posterior del cuaderno lo lee de disco en
lugar de volver a consultar el servidor, previa verificación de su
huella. Para redescargar hay que borrar los tres archivos del
congelamiento (.json, .sha256 y .meta.json), lo cual es una acción
deliberada y no puede ocurrir por accidente.

DESCARGA
--------
Servidor utilizado:
{servidor}

Generador:
{resultado_descarga["generador"]}

Fecha y hora de inicio:
{resultado_descarga["fecha_inicio"]}

Fecha y hora de finalización:
{resultado_descarga["fecha_fin"]}

Duración de la descarga (segundos):
{resultado_descarga["duracion_segundos"]}

Zona horaria:
America/Bogota

CRS:
{estaciones.crs}

BBOX DE CONSULTA
----------------
Oeste:
{bbox["oeste"]}

Sur:
{bbox["sur"]}

Este:
{bbox["este"]}

Norte:
{bbox["norte"]}

CRITERIO DE DELIMITACIÓN ESPACIAL
---------------------------------
{criterio_espacial}

LÍMITE OFICIAL (INSUMO MANUAL)
------------------------------
Archivo:
{ficha_limite["archivo"]}

Capa:
{ficha_limite["capa"]}

CRS original:
{ficha_limite["crs_original"]}

Geometrías en la capa original:
{ficha_limite["geometrias_originales"]}

Entidad productora:
{ficha_limite["entidad"]}

URL:
{ficha_limite["url"]}

Fecha de descarga del límite:
{ficha_limite["fecha_descarga_limite"]}

Forma de obtención:
{ficha_limite["obtencion"]}

AUDITORÍA
---------
Elementos recuperados de Overpass:
{auditoria["elementos_overpass"]}

Registros con coordenadas:
{auditoria["registros_con_coordenadas"]}

Registros sin coordenadas:
{auditoria["registros_sin_coordenadas"]}

Elementos con tipo no válido:
{auditoria["elementos_tipo_no_valido"]}

Registros finales:
{auditoria["registros_finales"]}

Registros excluidos por delimitación:
{auditoria["registros_excluidos_por_limite"]}

Estaciones en la frontera (menos de 50 m del borde municipal):
{auditoria["casos_frontera"]}

Registros con survey:date:
{auditoria["registros_con_survey_date"]}

Porcentaje retenido:
{auditoria["porcentaje_retenido"]} %

Porcentaje excluido:
{auditoria["porcentaje_excluido"]} %


ARCHIVOS
--------
GeoJSON:
{archivo_geojson}

Consulta Overpass:
{archivo_query}

Metadata:
{archivo_metadata}

Excluidas por límite:
{archivo_excluidas if n_excluidas else "ninguna estación excluida"}

Casos frontera:
{archivo_frontera if n_frontera else "ningún caso en la frontera"}
""".strip()

    with open(
        archivo_metadata,
        "w",
        encoding="utf-8",
    ) as archivo:

        archivo.write(metadata)

    return {
        "geojson":
            archivo_geojson,

        "consulta":
            archivo_query,

        "metadata":
            archivo_metadata,

        "excluidas":
            archivo_excluidas if n_excluidas else None,

        "casos_frontera":
            archivo_frontera if n_frontera else None,
    }

# 18. Validación del GeoJSON exportado

Escrito el archivo, validar_geojson_exportado lo vuelve a abrir y comprueba que conserve lo que debía conservar.

La verificación se realiza sobre el archivo en disco y no sobre el objeto en memoria. Esa distinción es lo que le da valor: comprueba el producto que el lector va a recibir, no el que el cuaderno creía estar produciendo. Entre ambos media la escritura, y es ahí donde pueden perderse cosas sin que nada lo advierta.

Es la aplicación directa del criterio adoptado tras el Bloque A —verificar por un camino distinto al que produjo el resultado— en el último punto donde todavía es posible detectar un problema.

## 18.1. El problema del viaje de ida y vuelta

La función auxiliar normalizar_campo_json atiende un comportamiento del controlador GeoJSON que conviene documentar porque no es evidente.

Los campos tags_json y sockets_json se escriben como cadenas de texto, según se explicó en la sección 11.5, porque el formato no admite objetos anidados. Al leer, sin embargo, el controlador de GDAL detecta las cadenas con forma de JSON y las deserializa por su cuenta. El resultado es que un mismo archivo devuelve texto o diccionario según la versión instalada de la biblioteca.

Esa inconsistencia haría que la validación pasara en una máquina y fallara en otra, sin que el archivo tuviera defecto alguno. La función absorbe la diferencia: acepta ambos tipos, además de los valores nulos y las cadenas vacías, y devuelve siempre un diccionario. El resto del cuaderno deja así de depender de la versión de la biblioteca.

Los tipos efectivamente devueltos se registran en el resumen, de modo que quede constancia de cómo se comportó el controlador en la ejecución concreta.

Ante un tipo no contemplado, la función lanza TypeError en lugar de devolver un diccionario vacío. Un valor de tipo inesperado indica un problema real que no debe absorberse silenciosamente.

## 18.2. Comprobaciones estructurales

Sobre el archivo releído se verifican cinco condiciones: que no esté vacío, que el número de registros coincida con el esperado, que conserve el sistema de referencia correcto, que contenga la columna osm_uid sin repeticiones, y que todas las geometrías sean puntos.

El conteo esperado. La comprobación recibe como argumento el número de registros que el inventario tenía en memoria y lo contrasta con los que el archivo contiene. Es la verificación de que la escritura no perdió filas.

El sistema de referencia. GeoJSON asume EPSG:4326 por especificación, pero el controlador puede escribir información de CRS y la lectura podría interpretarla de forma distinta. Comprobarlo cierra la posibilidad de que el archivo declare un sistema que no es el que sus coordenadas suponen.

La unicidad de osm_uid. Se verifica por tercera vez —tras el filtrado y tras la validación estructural— ahora sobre el archivo escrito. Las tres comprobaciones cubren etapas distintas del trayecto, y esta es la única que atestigua el estado del producto final.

## 18.3. Reconstrucción de los campos JSON

Se comprueba que ambos campos existan y que cada uno de sus valores pueda reconstruirse como diccionario.

Una cadena JSON truncada o mal codificada durante la escritura produciría un archivo legible como GeoJSON, con todas sus filas y geometrías correctas, pero con los conectores y las etiquetas irrecuperables. El defecto no se manifestaría hasta el Bloque B, al intentar la reconciliación de atributos, y para entonces el origen sería difícil de rastrear.

## 18.4. La verificación de contenido

La comprobación más importante es la última, y es de naturaleza distinta a las anteriores.

Que un campo JSON sea sintácticamente válido no garantiza que conserve lo que contenía. Una cadena vacía, un objeto sin claves o un diccionario parcialmente truncado son todos JSON válido. Las comprobaciones estructurales los aceptarían sin objeción.

La verificación contrasta dos cantidades calculadas por caminos independientes. n_tags se calculó durante la transformación, sobre el diccionario original recibido de Overpass, antes de cualquier serialización. El recuento de la derecha se calcula sobre el archivo ya escrito, deserializando la cadena y contando sus claves. Si alguna etiqueta se perdió en el trayecto, las cifras difieren.

Es exactamente la clase de comprobación cuya ausencia produjo el error crítico del Bloque A. Allí las distancias reportadas eran plausibles y nadie las cuestionó; el defecto solo apareció al recalcularlas por un método distinto del que las había generado. Aquí el mecanismo es el mismo: una cifra producida antes de exportar se contrasta contra la misma cifra recalculada después, y la coincidencia es lo que autoriza a afirmar que el archivo conserva la información.

La comprobación se aplica condicionalmente, solo si la columna n_tags existe, de modo que la función siga siendo utilizable sobre archivos que no la incluyan.

In [22]:
def normalizar_campo_json(valor):
    """
    Devuelve un dict a partir de un campo JSON leído de GeoJSON.

    El driver GeoJSON de GDAL detecta las cadenas con forma de JSON
    y las deserializa por su cuenta al leer, de modo que un mismo
    archivo devuelve str o dict según la versión instalada.
    Esta función absorbe esa diferencia para que el resto del
    cuaderno no dependa de ella.
    """

    if valor is None:
        return {}

    if isinstance(valor, dict):
        return valor

    if isinstance(valor, float) and pd.isna(valor):
        return {}

    if isinstance(valor, (str, bytes, bytearray)):

        texto = valor.strip() if isinstance(valor, str) else valor

        if not texto:
            return {}

        return json.loads(texto)

    raise TypeError(
        f"Tipo inesperado en campo JSON: {type(valor)}"
    )


def validar_geojson_exportado(
    archivo_geojson,
    ciudad,
    cantidad_esperada,
    codigo_dane_esperado=None,
):
    """
    Reabre el GeoJSON generado y comprueba
    que conserve la estructura esperada.

    codigo_dane_esperado, cuando se entrega, permite verificar
    sobre el archivo escrito que todas las estaciones se validaron
    contra el municipio del IGAC que correspondía. Es opcional
    para que la función siga siendo utilizable sobre archivos
    que no incorporen la columna.
    """

    gdf = gpd.read_file(
        archivo_geojson
    )

    if gdf.empty:

        raise ValueError(
            f"{ciudad}: el GeoJSON exportado está vacío."
        )

    if len(gdf) != cantidad_esperada:

        raise ValueError(
            f"{ciudad}: la cantidad de registros "
            f"del GeoJSON ({len(gdf)}) no coincide "
            f"con la cantidad esperada "
            f"({cantidad_esperada})."
        )

    if gdf.crs is None:

        raise ValueError(
            f"{ciudad}: el GeoJSON no tiene CRS."
        )

    if gdf.crs.to_string() != CRS_WGS84:

        raise ValueError(
            f"{ciudad}: CRS incorrecto en el GeoJSON."
        )

    if "osm_uid" not in gdf.columns:

        raise ValueError(
            f"{ciudad}: el GeoJSON no contiene osm_uid."
        )

    if gdf["osm_uid"].duplicated().any():

        raise ValueError(
            f"{ciudad}: existen osm_uid duplicados "
            "en el GeoJSON."
        )

    if not gdf.geometry.geom_type.eq(
        "Point"
    ).all():

        raise ValueError(
            f"{ciudad}: existen geometrías "
            "diferentes de Point."
        )

    # ------------------------------------------------------------------------
    # Delimitación aplicada
    # ------------------------------------------------------------------------
    # codigo_dane_limite viaja al inventario desde el GeoDataFrame
    # del límite a través del sjoin. Comprobarlo aquí, sobre el
    # archivo ya escrito, verifica dos cosas: que la constancia de
    # la delimitación sobrevivió a la exportación, y que todas las
    # estaciones se validaron contra un único municipio. Un archivo
    # con dos códigos distintos indicaría una mezcla de ciudades
    # que ninguna otra comprobación detectaría.

    if "codigo_dane_limite" not in gdf.columns:

        raise ValueError(
            f"{ciudad}: el GeoJSON no contiene "
            "codigo_dane_limite."
        )

    if gdf["codigo_dane_limite"].isna().any():

        raise ValueError(
            f"{ciudad}: existen registros sin "
            "codigo_dane_limite en el GeoJSON."
        )

    codigos_encontrados = set(
        gdf["codigo_dane_limite"]
        .astype(str)
        .unique()
    )

    if len(codigos_encontrados) != 1:

        raise ValueError(
            f"{ciudad}: se esperaba un único código DANE en el "
            f"inventario, se encontraron {codigos_encontrados}."
        )

    codigo_encontrado = codigos_encontrados.pop()

    if codigo_dane_esperado is not None:

        if codigo_encontrado != str(codigo_dane_esperado):

            raise ValueError(
                f"{ciudad}: el inventario se validó contra el "
                f"municipio {codigo_encontrado}, pero se esperaba "
                f"{codigo_dane_esperado}."
            )

    # ------------------------------------------------------------------------
    # Campos JSON: deben poder reconstruirse como diccionario
    # tras el viaje de ida y vuelta por el driver, sea que este
    # los devuelva como cadena o ya deserializados.
    # ------------------------------------------------------------------------

    tipos_devueltos = {}

    for columna in [
        "tags_json",
        "sockets_json",
    ]:

        if columna not in gdf.columns:

            raise ValueError(
                f"{ciudad}: el GeoJSON no contiene "
                f"{columna}."
            )

        try:

            parseados = gdf[columna].apply(
                normalizar_campo_json
            )

        except Exception as error:

            raise ValueError(
                f"{ciudad}: {columna} no se pudo reconstruir "
                f"tras la exportación: {error}"
            ) from error

        if not parseados.apply(
            lambda x: isinstance(x, dict)
        ).all():

            raise ValueError(
                f"{ciudad}: {columna} contiene valores que "
                "no son objetos JSON."
            )

        tipos_devueltos[columna] = sorted(
            {
                type(v).__name__
                for v in gdf[columna]
            }
        )

    # ------------------------------------------------------------------------
    # VERIFICACIÓN INDEPENDIENTE DE CONTENIDO
    #
    # No basta con que el JSON sea válido: hay que comprobar que
    # las etiquetas sobrevivieron. Se contrasta n_tags, calculado
    # antes de exportar, contra el conteo hecho sobre el archivo
    # ya escrito.
    # ------------------------------------------------------------------------

    if "n_tags" in gdf.columns:

        n_tags_recalculado = (
            gdf["tags_json"]
            .apply(normalizar_campo_json)
            .apply(len)
        )

        discrepancias = int(
            (
                n_tags_recalculado
                != gdf["n_tags"]
            ).sum()
        )

        if discrepancias > 0:

            raise ValueError(
                f"{ciudad}: en {discrepancias} registros el "
                "número de etiquetas del archivo no coincide "
                "con n_tags. Se perdió información al exportar."
            )

    return {
        "lectura_exitosa":
            True,

        "registros":
            len(gdf),

        "crs":
            gdf.crs.to_string(),

        "codigo_dane_limite":
            codigo_encontrado,

        "tipos_json_devueltos":
            tipos_devueltos,
    }

# 19. Pruebas de aceptación

ejecutar_pruebas_a2 reúne el conjunto de comprobaciones que el entregable debe superar y devuelve el registro de cada una.

Las pruebas se distinguen de las validaciones documentadas hasta aquí en su propósito. Las validaciones protegen el proceso: se ejecutan en medio del pipeline para impedir que un dato defectuoso avance. Las pruebas de aceptación certifican el producto: se ejecutan sobre el entregable terminado y su resultado se reporta.

El registro devuelto es parte del entregable. La revisión del Bloque A estableció que la comprobación debe quedar dentro del cuaderno y no en una revisión posterior, y este bloque cumple esa exigencia.

## 19.1. Comportamiento ante el fallo

Las pruebas interrumpen la ejecución mediante AssertionError en lugar de registrar un resultado negativo y continuar.

La consecuencia es que el registro devuelto solo contiene pruebas superadas: si alguna falla, no hay registro que devolver. La decisión evita que un entregable con pruebas fallidas llegue a producirse acompañado de una tabla que documente sus propios defectos como si fueran aceptables.

El uso de AssertionError y no de ValueError distingue en el mensaje una prueba de aceptación fallida de un error de datos, aunque ambos detengan el proceso.

## 19.2. Pruebas 1 a 6: verificación de las condiciones declaradas

Las seis primeras pruebas comprueban condiciones ya verificadas durante el proceso: validez de la consulta, inventario no vacío, unicidad de identificadores, sistema de referencia, tipo de geometría y archivo exportado.

La repetición es deliberada. Las validaciones intermedias se ejecutan sobre objetos que después se modifican; estas se ejecutan sobre el resultado final, y son las que autorizan a afirmar en el informe que el entregable cumple. Una condición verificada a mitad del proceso no dice nada sobre el estado en que quedó el producto.

La primera prueba registra los primeros doce caracteres de la huella SHA-256 de la consulta. El fragmento basta para identificar la ejecución y contrastarla contra el archivo de metadatos sin transcribir la huella completa.

## 19.3. Prueba 7: pertenencia al bbox recalculada

La séptima prueba comprueba que ninguna estación quede fuera del rectángulo de consulta, y lo hace comparando directamente las columnas latitud y longitud contra los cuatro límites declarados.

Lo relevante es el camino elegido. No se emplea la geometría ni ninguna operación espacial: se comparan números contra números. La comprobación es por tanto independiente tanto del bbox que se envió a Overpass como de la geometría construida durante la transformación.

Si el rectángulo escrito en la consulta hubiera diferido del declarado en la configuración, o si la geometría se hubiera construido a partir de coordenadas equivocadas, esta comparación lo detectaría.

## 19.4. Prueba 8: pertenencia al límite recalculada en metros

La octava prueba es la que responde de manera más directa al hallazgo de la revisión del Bloque A.

El filtrado espacial se realizó mediante sjoin con predicado intersects, en coordenadas geográficas. La prueba no repite esa operación. Reproyecta estaciones y límite a MAGNA-SIRGAS —sistema métrico oficial— y mide la distancia de cada punto al polígono, exigiendo que sea cero dentro de una tolerancia de un milímetro.

Son dos métodos distintos aplicados al mismo problema. El primero responde por predicado topológico en grados; el segundo, por medida de distancia en metros. Que ambos coincidan es lo que permite afirmar que el recorte se aplicó correctamente.

La analogía con el Bloque A es exacta. Allí las distancias reportadas eran plausibles y el error solo apareció cuando la tutoría las recalculó por una fórmula independiente. La lección que quedó formulada —cuando un resultado se ve bien, hay que verificarlo con un cálculo distinto al que lo produjo— está aquí implementada dentro del cuaderno, que es donde la revisión pidió que estuviera.

Sobre la tolerancia y las unidades. El margen de un milímetro absorbe el error de redondeo de la reproyección sin admitir desplazamientos con significado espacial. La medición se realiza en metros y no en grados porque una distancia calculada sobre coordenadas geográficas carece de unidades interpretables: fue precisamente el tipo de defecto que la revisión del Bloque A señaló, junto con la exigencia de trabajar en MAGNA-SIRGAS.

Cuando la prueba pasa, se registra la distancia máxima encontrada. La cifra documenta el margen con que se superó la comprobación, en lugar de limitarse a afirmar que se superó.

## 19.5. Prueba 9: diagnóstico de proximidad interna

La última prueba se distingue de todas las anteriores en que no puede fallar. No verifica una condición: mide una característica del inventario y la reporta.

Cuenta cuántas estaciones tienen otra estación de la misma fuente a menos de treinta metros, empleando la búsqueda del vecino más próximo en el sistema métrico y excluyendo cada registro de su propia comparación.

Su propósito es anticipar el Bloque B. La existencia de duplicados internos dentro de cada fuente es un problema conocido del proyecto, distinto del emparejamiento entre fuentes y que debe tratarse por separado. Dos elementos de OpenStreetMap a pocos metros pueden ser dos puntos de carga independientes en un mismo parqueadero, o el mismo dispositivo registrado dos veces. Esta prueba no resuelve la ambigüedad: la cuantifica y deja constancia de cuántos casos habrá que examinar.

## 19.6. Pruebas 10 a 13: balance, corte y frontera

Las cuatro últimas pruebas se incorporaron con el congelamiento del inventario.

La décima comprueba el balance de la transformación: los elementos recibidos deben repartirse sin pérdida entre transformados, descartados por falta de coordenadas y descartados por tipo. A diferencia de la cadena de conteos de la auditoría, que es una descomposición aritmética, este contraste puede fallar y por eso verifica algo.

La undécima comprueba que el servidor respetara el corte temporal solicitado. Es la salvaguarda contra el segundo fallo silencioso del servicio: un espejo sin datos históricos responde con el estado actual sin emitir error, y la única evidencia disponible es que el corte que declara sea posterior al pedido.

La duodécima recalcula la huella del archivo crudo sobre el archivo en disco y la contrasta con la registrada. Es lo que autoriza a afirmar que el inventario salió del crudo congelado y no de otra cosa.

La decimotercera no verifica: informa cuántas estaciones quedaron a menos de cincuenta metros del borde municipal. Es la contraparte de la advertencia que el IGAC declara sobre su propia capa. Una estación en esa franja queda clasificada por una geometría cuya exactitud a esa escala la entidad productora no garantiza, y corresponde reportarla como caso indeterminado antes que asignarla sin más.

In [23]:
def ejecutar_pruebas_a2(
    ciudad,
    query,
    estaciones,
    archivo_geojson,
    limite,
    bbox,
    fecha_corte=None,
    codigo_dane=None,
    auditoria_transformacion=None,
    resultado_descarga=None,
    casos_frontera=None,
):
    """
    Ejecuta las pruebas de aceptación del entregable A2.

    Las pruebas 7 y 8 son verificaciones independientes:
    comprueban el resultado con un cálculo distinto del que
    lo produjo, que es la corrección de método que dejó la
    revisión del Bloque A.
    """

    resultados = []

    def registrar(nombre, detalle=""):

        resultados.append({
            "ciudad": ciudad,
            "prueba": nombre,
            "resultado": "PASÓ",
            "detalle": detalle,
        })

    # ------------------------------------------------------------------------
    # Prueba 1 — Consulta Overpass
    # ------------------------------------------------------------------------

    validar_consulta_overpass(
        query,
        fecha_corte,
    )

    registrar(
        "Consulta Overpass válida",
        f"sha256={hash_consulta(query)[:12]}",
    )

    # ------------------------------------------------------------------------
    # Prueba 2 — Inventario no vacío
    # ------------------------------------------------------------------------

    if estaciones.empty:

        raise AssertionError(
            f"{ciudad}: inventario vacío."
        )

    registrar(
        "Inventario no vacío",
        f"{len(estaciones)} registros",
    )

    # ------------------------------------------------------------------------
    # Prueba 3 — osm_uid único
    # ------------------------------------------------------------------------

    if estaciones[
        "osm_uid"
    ].duplicated().any():

        raise AssertionError(
            f"{ciudad}: osm_uid duplicados."
        )

    registrar("osm_uid únicos")

    # ------------------------------------------------------------------------
    # Prueba 4 — CRS
    # ------------------------------------------------------------------------

    if estaciones.crs.to_string() != CRS_WGS84:

        raise AssertionError(
            f"{ciudad}: CRS incorrecto."
        )

    registrar("CRS EPSG:4326")

    # ------------------------------------------------------------------------
    # Prueba 5 — Geometrías
    # ------------------------------------------------------------------------

    if not estaciones.geometry.geom_type.eq(
        "Point"
    ).all():

        raise AssertionError(
            f"{ciudad}: geometrías inválidas."
        )

    registrar("Geometrías Point")

    # ------------------------------------------------------------------------
    # Prueba 6 — GeoJSON
    # ------------------------------------------------------------------------

    resultado_geojson = (
        validar_geojson_exportado(
            archivo_geojson,
            ciudad,
            len(estaciones),
            codigo_dane_esperado=codigo_dane,
        )
    )

    registrar(
        "GeoJSON exportado y validado",
        f"{resultado_geojson['registros']} registros, "
        f"municipio {resultado_geojson['codigo_dane_limite']}",
    )

    # ------------------------------------------------------------------------
    # Prueba 7 — VERIFICACIÓN INDEPENDIENTE: pertenencia al bbox
    #
    # Se recalcula a partir de las columnas latitud/longitud,
    # sin usar la geometría ni el sjoin.
    # ------------------------------------------------------------------------

    fuera_bbox = estaciones[
        (estaciones["longitud"] < bbox["oeste"])
        | (estaciones["longitud"] > bbox["este"])
        | (estaciones["latitud"] < bbox["sur"])
        | (estaciones["latitud"] > bbox["norte"])
    ]

    if len(fuera_bbox) > 0:

        raise AssertionError(
            f"{ciudad}: {len(fuera_bbox)} estaciones "
            "fuera del bbox de consulta."
        )

    registrar(
        "Verificación independiente: bbox",
        "recalculada desde latitud/longitud",
    )

    # ------------------------------------------------------------------------
    # Prueba 8 — VERIFICACIÓN INDEPENDIENTE: pertenencia al límite
    #
    # No se repite el sjoin que produjo el filtro. Se reproyecta a
    # MAGNA-SIRGAS y se mide la distancia en metros de cada punto
    # al polígono. Toda estación retenida debe estar a distancia
    # cero. Es el análogo de recalcular las distancias de A4.
    # ------------------------------------------------------------------------

    estaciones_m = estaciones.to_crs(
        CRS_METRICO
    )

    limite_m = limite.to_crs(
        CRS_METRICO
    )

    geometria_limite = limite_m.geometry.iloc[0]

    distancias = (
        estaciones_m.geometry
        .distance(geometria_limite)
    )

    tolerancia_metros = 0.001

    fuera_limite = distancias[
        distancias > tolerancia_metros
    ]

    if len(fuera_limite) > 0:

        raise AssertionError(
            f"{ciudad}: {len(fuera_limite)} estaciones "
            "fuera del límite oficial según el recálculo "
            f"métrico. Distancia máxima: "
            f"{fuera_limite.max():.2f} m."
        )

    registrar(
        "Verificación independiente: límite oficial",
        f"distancia máxima {distancias.max():.4f} m "
        f"en {CRS_METRICO}",
    )

    # ------------------------------------------------------------------------
    # Prueba 9 — Diagnóstico de proximidad
    #
    # No falla: informa. Estaciones a menos de 30 m entre sí
    # dentro de la misma fuente son candidatas a duplicado y
    # entran al Bloque B como casos difíciles.
    # ------------------------------------------------------------------------

    if len(estaciones_m) > 1:

        vecinos = estaciones_m.sjoin_nearest(
            estaciones_m,
            how="left",
            distance_col="dist_m",
            exclusive=True,
        )

        pares_cercanos = int(
            (vecinos["dist_m"] < 30).sum()
        )

    else:

        pares_cercanos = 0

    registrar(
        "Diagnóstico de proximidad interna",
        f"{pares_cercanos} registros con vecino "
        "a menos de 30 m",
    )

    # ------------------------------------------------------------------------
    # Prueba 10 — Balance de la transformación
    #
    # Los cuatro conteos se producen de forma independiente dentro
    # del mismo recorrido. Que sumen es una verificación real, a
    # diferencia de registros_excluidos_por_limite, que es una resta.
    # ------------------------------------------------------------------------

    if auditoria_transformacion is not None:

        a = auditoria_transformacion

        suma = (
            a["registros_transformados"]
            + a["registros_sin_coordenadas"]
            + a["elementos_tipo_no_valido"]
        )

        if suma != a["elementos_recibidos"]:

            raise AssertionError(
                f"{ciudad}: el balance de la transformación no cuadra. "
                f"Recibidos {a['elementos_recibidos']}, "
                f"suma de destinos {suma}. Hay elementos que el código "
                "no está contabilizando."
            )

        registrar(
            "Balance de la transformación",
            f"{a['elementos_recibidos']} = "
            f"{a['registros_transformados']} + "
            f"{a['registros_sin_coordenadas']} + "
            f"{a['elementos_tipo_no_valido']}",
        )

    # ------------------------------------------------------------------------
    # Prueba 11 — El corte temporal se respetó
    #
    # Un espejo sin datos históricos ignora [date:] y responde con el
    # estado actual sin emitir error. La evidencia no está en la
    # cabecera de la respuesta —timestamp_osm_base declara el estado
    # de la base de datos, no el de la consulta— sino en los datos:
    # ningún elemento puede haber sido editado después del corte.
    # ------------------------------------------------------------------------

    if fecha_corte and resultado_descarga is not None:

        edicion_mas_reciente = verificar_corte_respetado(
            resultado_descarga["elementos"],
            fecha_corte,
            ciudad,
        )

        registrar(
            "Corte temporal respetado",
            f"edición más reciente {edicion_mas_reciente}, "
            f"anterior al corte {fecha_corte}",
        )

    # ------------------------------------------------------------------------
    # Prueba 12 — Integridad del archivo crudo congelado
    #
    # Se recalcula la huella sobre el archivo en disco y se contrasta
    # con la registrada. Es lo que permite afirmar que el inventario
    # sale del crudo y no de otra cosa.
    # ------------------------------------------------------------------------

    if resultado_descarga is not None and resultado_descarga.get("ruta_crudo"):

        hash_recalculado = sha256_archivo(
            resultado_descarga["ruta_crudo"]
        )

        if hash_recalculado != resultado_descarga["hash_crudo"]:

            raise AssertionError(
                f"{ciudad}: la huella del archivo crudo no coincide "
                "con la registrada."
            )

        registrar(
            "Archivo crudo íntegro",
            f"sha256={hash_recalculado[:12]} sobre "
            f"{resultado_descarga['ruta_crudo']}",
        )

    # ------------------------------------------------------------------------
    # Prueba 13 — Casos indeterminados en la frontera
    #
    # No falla: informa. El IGAC declara que su capa no es apropiada
    # para la ubicación exacta de puntos, de modo que una estación
    # muy próxima al borde queda clasificada por una geometría cuya
    # exactitud a esa escala la entidad productora no garantiza.
    # ------------------------------------------------------------------------

    if casos_frontera is not None:

        if casos_frontera.empty:
            detalle = "ninguna estación a menos de 50 m del borde"
        else:
            detalle = (
                f"{len(casos_frontera)} estaciones a menos de 50 m "
                f"del borde; mínima "
                f"{casos_frontera['distancia_al_borde_m'].min():.2f} m"
            )

        registrar(
            "Casos indeterminados en la frontera",
            detalle,
        )

    return resultados

# 20. Ejecución del entregable A2

Esta celda ejecuta el proceso completo para cada área de estudio, encadenando las funciones definidas en las secciones anteriores. Es el único punto del cuaderno donde se produce trabajo efectivo: todo lo anterior son declaraciones y definiciones.

El proceso se organiza como un ciclo sobre AREAS_ESTUDIO, de modo que ambas ciudades atraviesan exactamente la misma secuencia de doce pasos. Que el tratamiento sea idéntico no es un detalle de implementación: garantiza que cualquier diferencia entre los resultados de Bogotá y Medellín provenga de los datos y no del procesamiento.

## 20.1. La secuencia

Los doce pasos se agrupan en cuatro fases.

Preparación y consulta. Se validan los bbox, se construye la consulta y se valida antes de enviarla. La huella de la consulta se imprime en este punto, de modo que quede visible en la salida de la ejecución y no solo en el archivo de metadatos.

Descarga y transformación. Se ejecuta la consulta contra los servidores de Overpass, se convierten los elementos recibidos en un GeoDataFrame y se validan las coordenadas resultantes.

El número de registros con coordenadas se captura en este punto, antes del recorte espacial, porque es el denominador que la auditoría empleará para calcular las proporciones. Capturarlo después haría imposible distinguir las pérdidas por defecto de la fuente de las exclusiones por pertenencia a otro municipio.

Delimitación y validación. Se carga el límite municipal, se aplica el recorte espacial y se valida el inventario resultante.

Auditoría, exportación y pruebas. Se consolida el balance del proceso, se producen los tres archivos del entregable y se ejecutan las pruebas de aceptación sobre lo producido.

## 20.2. El orden de exportación y pruebas

Las pruebas se ejecutan después de la exportación, no antes.

La razón es que la sexta prueba verifica el GeoJSON releyéndolo desde disco, lo que exige que el archivo exista. Y la lógica de fondo es la misma que sostiene toda la sección 18: lo que se certifica es el producto que el lector va a recibir, no el objeto que el cuaderno tenía en memoria.

Como las pruebas interrumpen la ejecución al fallar, el ciclo se detendría con los archivos ya escritos. Esa es la consecuencia deseada: los archivos quedan disponibles para diagnosticar qué falló, en lugar de desaparecer con el error.

## 20.3. La carga del límite dentro del ciclo

El paso de carga del límite requiere atención

La capa del IGAC es nacional. Cargarla sin indicar qué municipio seleccionar produciría, al disolverse, un polígono que abarca el país entero: un filtro espacial que existe formalmente y no filtra nada. Por eso cargar_limite_ciudad recibe aquí columna_codigo y codigo, y por eso la función bloquea la ejecución si no los recibe sobre una capa multirregistro.

El código del municipio se obtiene indexando codigos_dane con la clave del ciclo. Que las claves de AREAS_ESTUDIO y las de codigos_dane coincidan es lo que permite esa correspondencia automática, sin condicionales por ciudad.

Conviene señalar que esta carga duplica la que se ejecutó en la sección 13. La duplicación es deliberada: aquella se ejecuta antes de la descarga y detecta un problema con el archivo de límites sin haber consumido una consulta a Overpass; esta se ejecuta dentro del ciclo para que el proceso sea autónomo y no dependa de que la celda anterior se haya ejecutado en la misma sesión.

## 20.4. Acumulación de resultados

Tres estructuras recogen la salida del proceso.

resultados_a2 es un diccionario indexado por ciudad que consolida, para cada una, el conteo de registros, la identificación del servidor, las marcas temporales, la huella de la consulta, los resultados de validación y auditoría, la ficha del límite y las rutas de los archivos producidos.

auditoria_recorte y pruebas_a2 son listas que acumulan los registros de ambas ciudades en una sola estructura, aptas para convertirse en tabla comparativa sin manipulación adicional.

La separación responde a los dos usos previstos. El diccionario conserva la trazabilidad completa por ciudad; las listas alimentan los resúmenes tabulares de las celdas siguientes.

## 20.5. La salida impresa

Al cerrar cada ciudad se imprime un resumen con los conteos de la cadena de procesamiento, dos medidas de completitud de atributos y la procedencia de la descarga.

Las cifras de conectores declarados y de fecha de verificación merecen atención particular. No son diagnósticos del proceso sino observaciones sobre la fuente, y anticipan el problema que el Bloque B identificó como dominante en las fuentes abiertas colombianas: la limitación principal no es la precisión posicional sino la ausencia de los atributos que determinan la factibilidad de una ruta.

El corte de datos del servidor cierra el resumen. Con el corte ya fijado, la cifra impresa no es una instrucción sino una constatación, y conviene no confundirla con el corte de la consulta: declara hasta qué momento el servidor tenía sincronizado el mapa mundial, no qué estado histórico devolvió. La verificación del corte se hace contra las marcas temporales de los elementos, en la prueba 11.

In [30]:
resultados_a2 = {}

auditoria_recorte = []

pruebas_a2 = []


for ciudad, configuracion in AREAS_ESTUDIO.items():

    print("\n" + "=" * 80)

    print(
        f"A2 — PROCESANDO "
        f"{configuracion['nombre'].upper()}"
    )

    print("=" * 80)

    # Los códigos DANE están declarados en dos lugares
    # (AREAS_ESTUDIO y LIMITE_IGAC). Se exige que coincidan: es la
    # única manera de que la duplicación no se convierta en una
    # divergencia silenciosa.

    if (
        configuracion["codigo_dane"]
        != configuracion["limite"]["codigos_dane"][ciudad]
    ):
        raise AssertionError(
            f"{ciudad}: el código DANE de AREAS_ESTUDIO "
            f"({configuracion['codigo_dane']}) no coincide con el de "
            f"LIMITE_IGAC "
            f"({configuracion['limite']['codigos_dane'][ciudad]})."
        )

    # ------------------------------------------------------------------------
    # 20. Validar BBOX
    # ------------------------------------------------------------------------

    validar_bbox(
        configuracion["bbox"],
        configuracion["nombre"],
    )

    # ------------------------------------------------------------------------
    # 20.1. Construir consulta
    # ------------------------------------------------------------------------

    query = construir_consulta_overpass(
        bbox=configuracion["bbox"],
        fecha_corte=FECHA_CORTE_OSM,
    )

    validar_consulta_overpass(
        query,
        FECHA_CORTE_OSM,
    )

    print(
        "Huella de la consulta: "
        f"{hash_consulta(query)[:16]}"
    )

    # ------------------------------------------------------------------------
    # 20.2. Descargar desde Overpass
    # ------------------------------------------------------------------------

    resultado_descarga = (
        descargar_estaciones_osm(
            query=query,
            ciudad=ciudad,
            fecha_corte=FECHA_CORTE_OSM,
        )
    )

    elementos = (
        resultado_descarga["elementos"]
    )

    # ------------------------------------------------------------------------
    # 20.3. Transformar elementos
    # ------------------------------------------------------------------------

    (
        estaciones,
        auditoria_transformacion,
    ) = transformar_elementos_osm(
        elementos=elementos,
        ciudad=ciudad,
        fecha_descarga=(
            resultado_descarga[
                "fecha_fin"
            ]
        ),

        timestamp_osm_base=(
            resultado_descarga["timestamp_osm_base"]
        ),
        fecha_corte_osm=FECHA_CORTE_OSM,
    )

    registros_con_coordenadas = len(
        estaciones
    )

    # ------------------------------------------------------------------------
    # 20.4. Validar coordenadas
    # ------------------------------------------------------------------------

    validar_coordenadas(
        estaciones,
        ciudad,
    )

    # ------------------------------------------------------------------------
    # 20.5. Cargar límite oficial (insumo manual documentado)
    # ------------------------------------------------------------------------
    # La capa del IGAC es nacional: se debe indicar columna_codigo
    # y codigo para seleccionar el municipio. Sin ellos, la función
    # bloquea la ejecución en lugar de disolver los 1.105 municipios
    # en un solo polígono.

    (
        limite,
        ficha_limite,
    ) = cargar_limite_ciudad(
        archivo=configuracion["limite"]["archivo"],
        capa=configuracion["limite"]["capa"],
        nombre_ciudad=configuracion["nombre"],
        procedencia=(
            configuracion["limite"]["procedencia"]
        ),
        columna_codigo=(
            configuracion["limite"]["columna_codigo"]
        ),
        codigo=(
            configuracion["limite"]["codigos_dane"][ciudad]
        ),
    )

    # ------------------------------------------------------------------------
    # 20.6. Filtrar por límite oficial
    # ------------------------------------------------------------------------

    estaciones, estaciones_excluidas = filtrar_por_limite_oficial(
        estaciones,
        limite,
    )

    # ------------------------------------------------------------------------
    # 20.6b. Distancia al borde y casos indeterminados
    # ------------------------------------------------------------------------
    # Segunda consecuencia operativa de la restricción declarada por
    # el IGAC (sección 6.4). La columna viaja al GeoJSON.

    estaciones, casos_frontera = identificar_casos_frontera(
        estaciones,
        limite,
        umbral_metros=50.0,
    )

    # ------------------------------------------------------------------------
    # 20.6c. A qué municipio pertenecen las excluidas
    # ------------------------------------------------------------------------

    estaciones_excluidas = clasificar_excluidas_por_municipio(
        estaciones_excluidas,
        archivo=configuracion["limite"]["archivo"],
        capa=configuracion["limite"]["capa"],
        columna_codigo=configuracion["limite"]["columna_codigo"],
    )

    # ------------------------------------------------------------------------
    # 20.7. Validar inventario final
    # ------------------------------------------------------------------------

    validacion = validar_inventario_osm(
        estaciones,
        ciudad,
    )

    # ------------------------------------------------------------------------
    # 20.8. Construir auditoría
    # ------------------------------------------------------------------------

    auditoria = construir_auditoria(

        ciudad=ciudad,

        elementos_overpass=len(
            elementos
        ),

        registros_con_coordenadas=(
            registros_con_coordenadas
        ),

        registros_sin_coordenadas=(
            auditoria_transformacion[
                "registros_sin_coordenadas"
            ]
        ),

        registros_finales=len(
            estaciones
        ),

        elementos_tipo_no_valido=(
            auditoria_transformacion[
                "elementos_tipo_no_valido"
            ]
        ),

        casos_frontera=len(casos_frontera),

        registros_con_survey_date=(
            auditoria_transformacion[
                "registros_con_survey_date"
            ]
        ),
    )

    auditoria_recorte.append(
        auditoria
    )

    # ------------------------------------------------------------------------
    # 20.9. Exportar
    # ------------------------------------------------------------------------

    archivos = exportar_entregable_a2(

        estaciones=estaciones,

        query=query,

        resultado_descarga=(
            resultado_descarga
        ),

        ciudad=ciudad,

        auditoria=auditoria,

        bbox=configuracion["bbox"],

        criterio_espacial=(
            configuracion["criterio_espacial"]
        ),

        ficha_limite=ficha_limite,
        excluidas=estaciones_excluidas,

        casos_frontera=casos_frontera,
    )

    # ------------------------------------------------------------------------
    # 20.10. Ejecutar pruebas
    # ------------------------------------------------------------------------

    pruebas = ejecutar_pruebas_a2(

        ciudad=ciudad,

        query=query,

        estaciones=estaciones,

        archivo_geojson=archivos["geojson"],

        limite=limite,

        bbox=configuracion["bbox"],

        fecha_corte=FECHA_CORTE_OSM,
        
        codigo_dane=configuracion["limite"]["codigos_dane"][ciudad],

        auditoria_transformacion=auditoria_transformacion,

        resultado_descarga=resultado_descarga,

        casos_frontera=casos_frontera,
    )

    pruebas_a2.extend(
        pruebas
    )

    # ------------------------------------------------------------------------
    # 20.11. Guardar resultados
    # ------------------------------------------------------------------------

    resultados_a2[ciudad] = {

        "registros":
            len(estaciones),

        "servidor":
            resultado_descarga["servidor"],

        "fecha_inicio":
            resultado_descarga["fecha_inicio"],

        "fecha_fin":
            resultado_descarga["fecha_fin"],

        "duracion_segundos":
            resultado_descarga[
                "duracion_segundos"
            ],

        "timestamp_osm_base":
            resultado_descarga[
                "timestamp_osm_base"
            ],

        "hash_consulta":
            hash_consulta(query),

        "origen_datos":
            resultado_descarga.get("origen"),

        "ruta_crudo":
            resultado_descarga.get("ruta_crudo"),

        "hash_crudo":
            resultado_descarga.get("hash_crudo"),

        "excluidas":
            len(estaciones_excluidas),

        "casos_frontera":
            len(casos_frontera),

        "validacion":
            validacion,

        "auditoria":
            auditoria,

        "ficha_limite":
            ficha_limite,

        "archivos":
            archivos,
    }

    print(
        f"\n✓ {configuracion['nombre']} "
        "procesada correctamente."
    )

    print(
        f"Elementos recuperados: {len(elementos)}"
    )

    print(
        f"Registros con coordenadas: "
        f"{registros_con_coordenadas}"
    )

    print(
        f"Registros finales: {len(estaciones)}"
    )

    print(
        f"Excluidas por límite: {len(estaciones_excluidas)}"
    )

    if len(estaciones_excluidas):
        print(
            "  Municipios: "
            + ", ".join(
                sorted(
                    estaciones_excluidas["municipio_nombre"]
                    .dropna()
                    .unique()
                )
            )
        )

    print(
        f"Casos frontera (<50 m del borde): {len(casos_frontera)}"
    )

    print(
        f"Origen de los datos: {resultado_descarga.get('origen')}"
    )

    print(
        f"Con conectores declarados: "
        f"{auditoria_transformacion['registros_con_sockets']}"
    )

    print(
        f"Con check_date: "
        f"{auditoria_transformacion['registros_con_check_date']}"
    )

    print(
        f"Servidor: {resultado_descarga['servidor']}"
    )

    print(
        f"Corte de datos OSM: "
        f"{resultado_descarga['timestamp_osm_base']}"
    )


A2 — PROCESANDO BOGOTÁ D.C.
Huella de la consulta: 10c697bdb4ff20aa
  Corte verificado: edición más reciente 2026-07-14T14:57:26+00:00, anterior al corte solicitado (44 elementos).
Archivo congelado: A2_osm_crudo_Bogota_2026-08-29.json
  SHA-256 verificado: 883195e669b34f10
  44 elementos | corte OSM 2026-09-13T19:16:11Z
  Corte verificado: edición más reciente 2026-07-14T14:57:26+00:00, anterior al corte solicitado (44 elementos).

✓ Bogotá D.C. procesada correctamente.
Elementos recuperados: 44
Registros con coordenadas: 44
Registros finales: 42
Excluidas por límite: 2
  Municipios: Cota, Tenjo
Casos frontera (<50 m del borde): 0
Origen de los datos: archivo crudo congelado
Con conectores declarados: 12
Con check_date: 0
Servidor: https://overpass-api.de/api/interpreter
Corte de datos OSM: 2026-09-13T19:16:11Z

A2 — PROCESANDO MEDELLÍN
Huella de la consulta: fa67a3381af2e92a
  Corte verificado: edición más reciente 2026-01-23T04:31:48+00:00, anterior al corte solicitado (14 elemento

In [25]:
# ============================================================================
# Efecto de la sustitución del límite sobre el inventario
# ============================================================================
# La comparación geométrica de la sección 6.5 responde cuánto cambió
# el ÁREA de estudio. Esta responde cuánto cambió el INVENTARIO, que
# es la pregunta que importa. Se recorta el mismo inventario con el
# límite anterior y con el del IGAC, y se contrastan los conteos.
#
# LIMITES_SUSTITUIDOS se lee solo aquí y en las celdas de comparación:
# está fuera del circuito de análisis.

comparacion_limites = []

for ciudad, configuracion in AREAS_ESTUDIO.items():

    inventario = gpd.read_file(
        f"A2_estaciones_OSM_{ciudad}.geojson"
    )

    crudo = rutas_crudo(ciudad)["crudo"]

    with open(crudo, encoding="utf-8") as archivo:
        data = json.load(archivo)

    previo, _ = transformar_elementos_osm(
        elementos=data["elements"],
        ciudad=ciudad,
        fecha_descarga="comparacion",
    )

    sustituido = LIMITES_SUSTITUIDOS[ciudad]

    limite_anterior = gpd.read_file(
        sustituido["archivo"],
        layer=sustituido["capa"],
    ).to_crs(CRS_WGS84)

    limite_anterior = gpd.GeoDataFrame(
        {"codigo_dane_limite": ["anterior"]},
        geometry=[limite_anterior.union_all()],
        crs=CRS_WGS84,
    )

    con_anterior, _ = filtrar_por_limite_oficial(
        previo,
        limite_anterior,
    )

    uids_igac = set(inventario["osm_uid"])
    uids_anterior = set(con_anterior["osm_uid"])

    comparacion_limites.append({
        "ciudad": configuracion["nombre"],
        "descargadas": len(previo),
        "con_limite_anterior": len(uids_anterior),
        "con_limite_IGAC": len(uids_igac),
        "solo_con_el_anterior": len(uids_anterior - uids_igac),
        "solo_con_el_IGAC": len(uids_igac - uids_anterior),
    })

comparacion_limites = pd.DataFrame(comparacion_limites)

display(comparacion_limites)

print(
    "\nLectura: si las dos últimas columnas son cero, la sustitución "
    "cambió la procedencia y la independencia del insumo sin mover "
    "una sola estación del inventario."
)

,ciudad,descargadas,con_limite_anterior,con_limite_IGAC,solo_con_el_anterior,solo_con_el_IGAC
0,Bogotá D.C.,44,42,42,0,0
1,Medellín,14,11,11,0,0



Lectura: si las dos últimas columnas son cero, la sustitución cambió la procedencia y la independencia del insumo sin mover una sola estación del inventario.


# Inspección del inventario exportado

Muestra las primeras filas de cada archivo producido, con las dos columnas que identifican la delimitación aplicada: `ciudad`, que es la etiqueta de trabajo, y `codigo_dane_limite`, que registra contra qué municipio del IGAC se validó cada estación.

No es una validación —esas se ejecutan en la sección 18 y en las pruebas de aceptación—, sino una lectura directa del producto, para constatar a simple vista que el inventario contiene lo que debe contener.

In [26]:
for clave in AREAS_ESTUDIO:

    inventario = gpd.read_file(
        f"A2_estaciones_OSM_{clave}.geojson"
    )

    print(f"\n=== {clave} — {len(inventario)} registros ===")

    display(
        inventario[
            ["osm_uid", "ciudad", "codigo_dane_limite", "nombre"]
        ].head(5)
    )


=== Bogota — 42 registros ===


,osm_uid,ciudad,codigo_dane_limite,nombre
0,node/6876702735,Bogota,11001,EDS Terpel - Cruz Roja
1,node/6876710563,Bogota,11001,Estación de carga de Unicentro
2,node/6876729187,Bogota,11001,Autogermana 128
3,node/6876751154,Bogota,11001,Estación de carga El Retiro
4,node/6876751240,Bogota,11001,Estación de carga Aparcar 114



=== Medellin — 11 registros ===


,osm_uid,ciudad,codigo_dane_limite,nombre
0,node/3811577575,Medellin,05001,Punto de carga Primax Nutibara
1,node/5188775821,Medellin,05001,NaN
2,node/9902522850,Medellin,05001,Punto de carga Voltex El Encierro (Medellín)
3,node/10697442797,Medellin,05001,Estacion de gasolina
4,node/13009470546,Medellin,05001,Punto de carga Voltex Autopista Norte


# 21. Resumen de resultados

Esta celda consolida las auditorías de ambas ciudades en una tabla comparativa. auditoria_recorte acumuló un diccionario por ciudad durante la ejecución, y esa lista se convierte directamente en DataFrame sin transformación intermedia.

## 21.1. Selección y orden de las columnas

Se seleccionan explícitamente las nueve columnas y se fija su orden, en lugar de mostrar el DataFrame tal como resulta de la conversión.

El orden reproduce la cadena de procesamiento: se parte de los elementos que Overpass entregó, se descuentan los descartes de la transformación, se llega al inventario previo al recorte, se aplica la exclusión por límite municipal y se cierra con las proporciones. Leída de izquierda a derecha, la fila cuenta qué le ocurrió al inventario de cada ciudad.

La selección explícita cumple además una función de estabilidad: si en el futuro se añadieran campos a la auditoría, la tabla conservaría su estructura y el cambio exigiría una decisión deliberada sobre qué mostrar.

## 21.2. Conteos absolutos antes que proporciones

Las siete primeras columnas son conteos absolutos y solo las dos últimas son porcentajes.

Responde a una de las reglas de trabajo del equipo: se reportan conteos absolutos y no solo porcentajes. En inventarios del orden de las decenas de estaciones, una proporción aislada es engañosa —un mismo porcentaje puede corresponder a una estación o a varias— y la revisión del Bloque A señaló precisamente ese defecto en los porcentajes del semáforo, calculados sobre un denominador que no era el inventario real.

Presentar ambas cifras juntas permite que el lector reconstruya el denominador sin necesidad de confiar en él.

## 21.3. Lectura de la tabla

La comparación entre ciudades es el uso principal de esta salida, y hay dos contrastes previsibles.

Las exclusiones por límite. Deberían diferir de manera apreciable, y por una razón geométrica documentada. El bbox de Bogotá cubre 338,2 km² fuera del Distrito, repartidos entre once municipios vecinos; el de Medellín cubre 268,6 km² correspondientes al Área Metropolitana. La proporción excluida en cada ciudad informa sobre la densidad de infraestructura de carga en esos bordes.

Los descartes por falta de coordenadas. Miden calidad de registro en OpenStreetMap, no geometría del área consultada. Si difieren entre ciudades, la diferencia habla de las prácticas de mapeo de cada comunidad local.

Ambas columnas deben interpretarse por separado, que es la razón por la cual la auditoría las mantiene desagregadas en lugar de reportar una cifra única de pérdidas.

In [27]:
resumen_a2 = pd.DataFrame(
    auditoria_recorte
)

columnas_resumen = [

    "ciudad",

    "elementos_overpass",

    "registros_con_coordenadas",

    "registros_sin_coordenadas",

    "elementos_tipo_no_valido",

    "registros_finales",

    "registros_excluidos_por_limite",

    "porcentaje_retenido",

    "porcentaje_excluido",
]

resumen_a2 = resumen_a2[
    columnas_resumen
]

display(
    resumen_a2
)

,ciudad,elementos_overpass,registros_con_coordenadas,registros_sin_coordenadas,elementos_tipo_no_valido,registros_finales,registros_excluidos_por_limite,porcentaje_retenido,porcentaje_excluido
0,Bogota,44,44,0,0,42,2,95.45,4.55
1,Medellin,14,14,0,0,11,3,78.57,21.43


# 22. Resumen de pruebas

Esta celda presenta el registro de las pruebas de aceptación ejecutadas sobre ambas ciudades. Pruebas_a2 acumuló los resultados de las trece pruebas por ciudad, y la lista se convierte directamente en tabla.

A diferencia del resumen anterior, no se seleccionan ni se ordenan columnas: el registro se muestra tal como lo construyó ejecutar_pruebas_a2, con sus cuatro campos —ciudad, prueba, resultado y detalle.

## 22.1. Una tabla que solo puede contener aprobaciones

La columna resultado toma el valor "PASÓ" en todas las filas, y no puede tomar otro.

La razón está en el diseño documentado en la sección 19.1: las pruebas interrumpen la ejecución mediante AssertionError al fallar, en lugar de registrar un resultado negativo. Si alguna hubiera fallado, el ciclo de ejecución se habría detenido y esta celda no tendría datos que mostrar.

La tabla no reporta, por tanto, qué proporción de pruebas se superó. Constata que se superaron todas, y su existencia misma es la evidencia. Un lector que vea esta tabla poblada sabe que el entregable completó el conjunto de verificaciones.

Esa constatación tiene un correlato operativo: el número de filas debe ser el producto del número de pruebas por el número de ciudades. Un total menor indicaría que el ciclo no llegó a procesar todas las áreas de estudio.

## 22.2. El valor de la columna de detalle

El campo detalle es lo que distingue este registro de una simple lista de verificaciones marcadas.

Varias pruebas consignan ahí la cifra con que se superaron: el fragmento de la huella de la consulta, el número de registros del inventario, la cantidad de registros releídos del GeoJSON, la distancia máxima medida en el recálculo métrico, y el conteo de estaciones con vecino próximo.

La diferencia entre afirmar que una prueba pasó y documentar con qué margen pasó es sustantiva. La octava prueba ilustra el punto: registrar que la distancia máxima al polígono fue de un valor concreto en MAGNA-SIRGAS permite constatar cuán holgadamente se cumplió la condición, mientras que un simple indicador de aprobación ocultaría un resultado apenas dentro de la tolerancia.

## 22.3. La fila que no verifica nada

La novena prueba aparece en esta tabla con el mismo valor "PASÓ" que las demás, pero su naturaleza es distinta y conviene tenerlo presente al leerla.

La decimotercera tiene la misma naturaleza. Cuenta cuántas estaciones quedaron a menos de cincuenta metros del borde municipal y registra el resultado sin condicionar la aprobación. En esta ejecución el conteo fue cero en ambas ciudades, de modo que ninguna estación del inventario cayó en la franja donde la restricción declarada por el IGAC volvería indeterminada la asignación.

No comprueba una condición: cuenta cuántas estaciones tienen otra a menos de treinta metros. Su valor de aprobación es incondicional, y lo que aporta es la cifra registrada en el detalle, que anticipa cuántos casos de posible duplicación interna deberá examinar el Bloque B.

Leída sin esa advertencia, la fila sugiere que se verificó algo que no se verificó.

## 22.4. Función de este registro dentro del entregable

La revisión del Bloque A estableció que la verificación independiente debe formar parte del cuaderno y no de una revisión posterior, y señaló que entregar el código junto con las salidas es lo que permite que un error se encuentre en una reunión de tutoría y no en una sustentación.

Esta tabla es la materialización de ese criterio. Muestra, para cada ciudad, qué se comprobó y con qué resultado, en el mismo documento que produjo los datos y de manera que quien lo lea pueda reejecutarlo.

In [28]:
resumen_pruebas_a2 = pd.DataFrame(
    pruebas_a2
)

display(
    resumen_pruebas_a2
)

,ciudad,prueba,resultado,detalle
0,Bogota,Consulta Overpass válida,PASÓ,sha256=10c697bdb4ff
1,Bogota,Inventario no vacío,PASÓ,42 registros
2,Bogota,osm_uid únicos,PASÓ,
3,Bogota,CRS EPSG:4326,PASÓ,
4,Bogota,Geometrías Point,PASÓ,
5,Bogota,GeoJSON exportado y validado,PASÓ,"42 registros, municipio 11001"
6,Bogota,Verificación independiente: bbox,PASÓ,recalculada desde latitud/longitud
7,Bogota,Verificación independiente: límite oficial,PASÓ,distancia máxima 0.0000 m en EPSG:9377
8,Bogota,Diagnóstico de proximidad interna,PASÓ,2 registros con vecino a menos de 30 m
9,Bogota,Balance de la transformación,PASÓ,44 = 44 + 0 + 0


# 23. Validación final del entregable A2

La última celda comprueba que el entregable esté completo antes de darlo por cerrado. Se ejecuta sobre las estructuras acumuladas, no sobre los datos, y su función es distinta de todas las validaciones anteriores.

Las validaciones previas comprueban el contenido: que las coordenadas sean válidas, que el archivo conserve lo escrito, que las estaciones caigan dentro del municipio. Esta comprueba la cobertura del proceso: que se haya hecho todo lo que se debía hacer.

Es la diferencia entre verificar que lo producido esté bien y verificar que se haya producido todo. Un cuaderno puede ejecutar correctamente sobre una sola ciudad y dejar la otra sin procesar, sin que ninguna de las comprobaciones anteriores lo advierta.

## 23.1. Cobertura de las áreas de estudio

La primera comprobación contrasta el número de entradas acumuladas en resultados_a2 contra el número de áreas declaradas en AREAS_ESTUDIO.

Detecta el caso en que el ciclo se hubiera interrumpido tras procesar una ciudad. Si la ejecución falló en la segunda, el cuaderno mostraría resultados de la primera, tablas pobladas y salidas de apariencia normal; nada indicaría que falta la mitad del entregable salvo esta comprobación.

Que el conteo se haga contra AREAS_ESTUDIO y no contra un número fijo importa: si el proyecto incorporara una tercera ciudad, la verificación se ajustaría sola.

## 23.2. Resumen no vacío

La segunda comprobación verifica que la tabla de auditoría tenga contenido.

Es redundante respecto de la anterior en el flujo actual, puesto que ambas estructuras se pueblan en el mismo ciclo. Cubre, sin embargo, el caso en que la celda de resumen no se hubiera ejecutado o se hubiera ejecutado sobre una estructura previamente reinicializada, situación posible en un cuaderno donde las celdas pueden correrse en desorden.

## 23.3. La comprobación que no puede fallar

La tercera comprobación verifica que todas las pruebas registren aprobación.

Su resultado está determinado de antemano, y conviene declararlo. Como las pruebas de aceptación interrumpen la ejecución mediante AssertionError al fallar, una prueba fallida habría detenido el ciclo de la sección 20 y esta celda no llegaría a ejecutarse. La columna resultado solo puede contener aprobaciones.

La verificación no es por ello inútil, pero su valor es distinto del que aparenta. No detecta pruebas fallidas: detecta que la estructura de la tabla sea la esperada. Si una modificación futura hiciera que las pruebas registraran fallos en lugar de interrumpir, esta comprobación pasaría a tener efecto real sin necesidad de reescribirla.

Documentarla como salvaguarda anticipada, y no como verificación activa, evita atribuirle una función que hoy no cumple.

## 23.4. La constancia de cierre

Superadas las tres comprobaciones, se imprime el bloque de cierre con el número de ciudades procesadas, el de pruebas ejecutadas y el de pruebas aprobadas.

Las dos últimas cifras deben coincidir, por la razón expuesta arriba. Su coincidencia no demuestra que las pruebas se superaran —eso lo demuestra el hecho de que la ejecución llegara hasta aquí— sino que el registro está completo y no perdió filas.

El bloque cumple una función práctica en un cuaderno extenso: ofrece en un solo lugar la constancia de que el entregable se produjo íntegro, sin necesidad de recorrer las salidas anteriores.

In [29]:
if len(resultados_a2) != len(
    AREAS_ESTUDIO
):

    raise AssertionError(
        "No se procesaron todas las áreas de estudio."
    )


if resumen_a2.empty:

    raise AssertionError(
        "El resumen de A2 está vacío."
    )


if not (
    resumen_pruebas_a2[
        "resultado"
    ] == "PASÓ"
).all():

    raise AssertionError(
        "Una o más pruebas de aceptación de A2 fallaron."
    )


print("=" * 80)

print(
    "✓ A2 COMPLETADO CORRECTAMENTE"
)

print("=" * 80)

print(
    f"Ciudades procesadas: "
    f"{len(resultados_a2)}"
)

print(
    f"Pruebas ejecutadas: "
    f"{len(resumen_pruebas_a2)}"
)

print(
    f"Pruebas aprobadas: "
    f"{(
        resumen_pruebas_a2['resultado'] == 'PASÓ'
    ).sum()}"
)

print("=" * 80)

✓ A2 COMPLETADO CORRECTAMENTE
Ciudades procesadas: 2
Pruebas ejecutadas: 26
Pruebas aprobadas: 26
